# Studio Zero · private engine
Runs the AI behind your Studio Zero canvas: images and AI edits, transparent images, background removal,
prompt help, upscaling, poses, voice, transcription, and optionally music and video. The canvas stays on your computer.

**How to use**
1. **Runtime → Run all.** The notebook asks for an A100 with High-RAM (change it in *Runtime → Change runtime type*).
2. Allow Google Drive when asked. Models are saved there once, so later sessions download nothing.
3. When step 5 shows **Studio Zero is ready**, click **Open Studio Zero** or paste the address and key into the studio.
4. When you finish, run **Stop and save**.

The first session takes about 30 minutes (downloads). Later sessions are ready in about 5 minutes.
Everything below is optional. Details, sizes and model licences are under **Reference** at the end.

## Settings

In [ ]:
#@title ⚙️ Settings: tick what you need { display-mode: "form" }
#@markdown **Features** (each model downloads once, to Drive)
ENABLE_REFERENCES = True  #@param {type:"boolean"}
#@markdown ↳ Reference images: guide new images with up to three pictures, and change poses
ENABLE_BACKGROUND_REMOVAL = True  #@param {type:"boolean"}
#@markdown ↳ Remove background on any layer (BiRefNet, 1 GB)
ENABLE_ASSISTANT = True  #@param {type:"boolean"}
#@markdown ↳ Improve prompt and Describe image (Qwen3-VL-2B, 4 GB)
ENABLE_UPSCALE = True  #@param {type:"boolean"}
#@markdown ↳ Upscale layers ×2 or ×4 (Real-ESRGAN, 67 MB)
ENABLE_POSE = True  #@param {type:"boolean"}
#@markdown ↳ Detect a body pose in the pose editor (DWPose, 240 MB)
ENABLE_VOICE = True  #@param {type:"boolean"}
#@markdown ↳ Text to speech and voice cloning, English (Chatterbox Turbo, 3 GB)
ENABLE_TRANSCRIBE = True  #@param {type:"boolean"}
#@markdown ↳ Speech to text and subtitles (faster-whisper, 145 MB)
ENABLE_MUSIC = False  #@param {type:"boolean"}
#@markdown ↳ Songs with vocals (MiniMax Music 3, 28 GB, needs a 24 GB GPU)
ENABLE_VIDEO = False  #@param {type:"boolean"}
#@markdown ↳ Video with sound (LTX-2.5, 44 GB, needs an A100/H100 and a Hugging Face token below)

#@markdown ---
#@markdown **Hugging Face token** (needed for video: the LTX-2.5 files are gated)
HF_TOKEN = ""  #@param {type:"string"}
#@markdown ↳ A *read* token from huggingface.co/settings/tokens, after accepting the licence on huggingface.co/Lightricks/LTX-2.5.
#@markdown Leave empty to use the Colab secret `HF_TOKEN` (🔑 in the left bar) instead. A token typed here is saved
#@markdown in this notebook, so clear it before sharing the notebook. It is never printed.

#@markdown ---
#@markdown **Speed and quality**
ENGINE = "auto"  #@param ["auto", "fast", "sdcpp"]
#@markdown ↳ auto: the 4-step fast engine on GPUs with 22 GB+ and 45 GB+ RAM, otherwise stable-diffusion.cpp
MODEL_SIZE = "Q4_K_M"  #@param ["Q4_K_M", "Q5_K_M", "Q6_K", "Q8_0", "BF16"]
#@markdown ↳ Image model file: Q4_K_M 4.6 GB · Q5_K_M 5.2 · Q6_K 5.9 · Q8_0 7.6 · BF16 14.2 (bigger = finer detail)
OFFLOAD = "auto"  #@param ["auto", "on", "off"]
#@markdown ↳ auto: keep weights on the GPU whenever they fit
CHECKS = "quick"  #@param ["quick", "full", "off"]
#@markdown ↳ Step 6 checks: quick = one image · full = every feature you ticked (a few minutes)

#@markdown ---
#@markdown **Google Drive cache**
CACHE_ON_DRIVE = True  #@param {type:"boolean"}
#@markdown ↳ Keep models in *My Drive/StudioZero* and reuse them every session
COPY_WEIGHTS_TO_LOCAL = False  #@param {type:"boolean"}
#@markdown ↳ Copy models to the local disk first (only if loading from Drive stalls)
UPDATE_MODELS = False  #@param {type:"boolean"}
#@markdown ↳ Check Hugging Face for newer model versions
REDOWNLOAD_MODELS = False  #@param {type:"boolean"}
#@markdown ↳ Download every model again (a damaged file). Untick afterwards
REBUILD_ENGINE = False  #@param {type:"boolean"}
#@markdown ↳ Recompile stable-diffusion.cpp. Untick afterwards


In [ ]:
#@title 🔧 Advanced settings (model sources, rarely changed) { display-mode: "form" }
FLASH_ATTENTION = True  #@param {type:"boolean"}
SAGE_ATTENTION = False  #@param {type:"boolean"}
ENGINE_SWAP = True  #@param {type:"boolean"}
#@markdown ↳ Engine swap: unload a model when the next one does not fit (untick only with memory for all of them)
VIDEO_FORMAT = "mp4"  #@param ["mp4", "webm"]
DRIVE_DIR = '/content/drive/MyDrive/StudioZero'
DIFFUSION_REPO = 'abenzerps/Qwen-Image-2.1-GGUF'
DIFFUSION_FILE = f'qwen-image-2.1-UC-{MODEL_SIZE}.gguf'  # any file on the repo's main branch
COMPANION_REPO = 'Qwen/Qwen-Image-2.1'                  # text encoder, VAE and configs for the fast engine
FAST_REPO = 'alibaba-pai/Qwen-Image-2.1-Fun-Acc-LoRAs'  # 4-step adapter + its loader code
FAST_FILE = 'models/Qwen-Image-2.1-Fun-Acc-4Step.safetensors'
MATTING_REPO = 'onnx-community/BiRefNet-ONNX'
MATTING_FILE = 'onnx/model.onnx'  # 'onnx/model_fp16.onnx' halves the download (less tested)
ASSISTANT_REPO = 'Qwen/Qwen3-VL-2B-Instruct'
WHISPER_REPO = 'Systran/faster-whisper-base'
VOICE_REPO = 'ResembleAI/chatterbox-turbo'
MUSIC_REPO = 'MiniMaxAI/MiniMax-Music3'
UPSCALE_MODEL = ('Comfy-Org/Real-ESRGAN_repackaged', 'RealESRGAN_x4plus.safetensors')
POSE_MODEL = ('yzd-v/DWPose', 'dw-ll_ucoco_384.onnx')
POSE_DETECTOR = ('hr16/yolox-onnx', 'yolox_m.onnx')
VIDEO_DIFFUSION = ('vantagewithai/LTX-2.5-GGUF', 'distilled/ltx-2.5-22b-distilled-transformer-Q4_K_M.gguf')
VIDEO_STEPS, VIDEO_GUIDANCE = 8, 1.0   # distilled model; for a 'dev/...' file use about 30 steps and guidance 3
VIDEO_TEXT_ENCODER = ('Lightricks/LTX-2.5', 'text_encoders/gemma4-12b-with-proj-ltx-2.5-bf16.safetensors')
VIDEO_VAE = ('Lightricks/LTX-2.5', 'vae/ltx-2.5-video-vae-conv-bf16.safetensors')   # the conv VAE is required
VIDEO_AUDIO_VAE = ('Lightricks/LTX-2.5', 'vae/ltx-2.5-audio-vae-bf16.safetensors')


## Start

In [ ]:
#@title 1 · Prepare the GPU, packages and Drive { display-mode: "form" }
import os, sys, subprocess, shutil, time, json, secrets, threading
from pathlib import Path

def ok(message):
    print('✓', message)

def pip(*packages, flags=()):
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *flags, *packages], capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-3000:], result.stderr[-3000:])
        raise RuntimeError('Installing ' + ' '.join(packages) + ' failed (pip output above).')

t0 = time.monotonic()
def gpu_query(field):
    return subprocess.run(['nvidia-smi', f'--query-gpu={field}', '--format=csv,noheader,nounits'],
                          capture_output=True, text=True, check=True).stdout.splitlines()[0].strip()
GPU_NAME = gpu_query('name')
GPU_GIB = float(gpu_query('memory.total')) / 1024
GPU_ARCH = gpu_query('compute_cap').replace('.', '')
RAM_GIB = int(open('/proc/meminfo').read().split('MemTotal:')[1].split()[0]) / 2**20
FAST = ENGINE == 'fast' or (ENGINE == 'auto' and GPU_GIB >= 22 and RAM_GIB >= 45)
ok(f"{GPU_NAME} · {GPU_GIB:.0f} GB GPU · {RAM_GIB:.0f} GB RAM → "
   f"{'fast engine (4 steps per image)' if FAST else 'stable-diffusion.cpp engine'}")

pip('fastapi>=0.115,<1', 'uvicorn>=0.34,<1', 'pillow>=11,<13', 'httpx>=0.28,<1', 'huggingface_hub>=0.30,<2', 'hf_xet')
if ENABLE_BACKGROUND_REMOVAL or ENABLE_TRANSCRIBE or ENABLE_POSE:
    # ONNX Runtime 1.27+ is built for CUDA 13, 1.21-1.26 for CUDA 12; match the CUDA that torch brings.
    TORCH_CUDA = subprocess.run([sys.executable, '-c', 'import torch; print(torch.version.cuda or "")'],
                                capture_output=True, text=True).stdout.strip()
    pip('onnxruntime-gpu>=1.21,<1.27' if TORCH_CUDA.startswith('12') else 'onnxruntime-gpu>=1.27,<2', 'numpy')
if FAST or ENABLE_ASSISTANT or ENABLE_VOICE or ENABLE_MUSIC:
    # Same pins as a public Space that runs the UC GGUF with Fun-Acc in diffusers.
    pip('diffusers @ git+https://github.com/huggingface/diffusers.git@9f1246971270c84dcbe71233edb7a519596a5d02',
        'transformers==5.17.0', 'accelerate', 'peft', 'gguf==0.19.0', 'sentencepiece')
    # Colab's preinstalled torchao is older than current peft expects and breaks the diffusers LoRA loader.
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], capture_output=True)
if ENABLE_TRANSCRIBE:
    # --no-deps: faster-whisper would pull the CPU onnxruntime over the GPU build installed above.
    pip('faster-whisper==1.2.1', flags=['--no-deps'])
    pip('ctranslate2>=4.6,<5', 'av>=11')
if ENABLE_UPSCALE:
    pip('spandrel==0.4.2')
if ENABLE_POSE:
    # --no-deps: rtmlib would pull the CPU onnxruntime over the GPU build; Colab already has OpenCV.
    pip('rtmlib==0.0.16', flags=['--no-deps'])
if ENABLE_VOICE:
    # --no-deps: chatterbox-tts pins torch 2.6, transformers 5.2 and safetensors 0.5, which would break the stack.
    pip('chatterbox-tts==0.1.7', flags=['--no-deps'])
    pip('resemble-perth>=1.0.1', 's3tokenizer', 'conformer==0.3.2', 'pyloudnorm', 'omegaconf', 'librosa', 'einops')
ok(f'Packages installed ({time.monotonic() - t0:.0f}s)')

ROOT = Path('/content/studio-zero')
ROOT.mkdir(exist_ok=True)
CACHE = None
if CACHE_ON_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE = Path(DRIVE_DIR); CACHE.mkdir(parents=True, exist_ok=True)
    ok(f'Google Drive connected · models are kept in {CACHE}')
else:
    print('• Drive cache is off: models download again every session.')


In [ ]:
#@title 2 · Install the Studio Zero API (bundled source) { display-mode: "form" }
from pathlib import Path
import sys
SOURCES = {'backend/__init__.py': '"""Studio Zero\'s replaceable private image service."""\n', 'backend/app.py': '"""Single-user, authenticated job API. Run with one uvicorn worker."""\nimport hmac\nimport json\nimport os\nimport secrets\nimport uuid\nfrom concurrent.futures import ThreadPoolExecutor\nfrom contextlib import asynccontextmanager\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom threading import Event, RLock\nfrom fastapi import Depends, FastAPI, Header, HTTPException, Request\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import FileResponse\nfrom PIL import Image, PngImagePlugin\nfrom .assistant import DemoPromptHelper, PromptHelper\nfrom .audio import (DemoMusicRunner, DemoSpeaker, DemoTranscriber, MusicRunner, Speaker, Transcriber, audio_format,\n                    finish_audio)\nfrom .engines import EngineSet\nfrom .vision import DemoPoseDetector, DemoUpscaler, PoseDetector, Upscaler, check_upscale_size\nfrom .matting import default_matter\nfrom .runner import Cancelled, DemoRunner, DemoVideoRunner, QwenRunner, ResidentQwenRunner, ResidentVideoRunner, SD_REVISION\nfrom .schema import (DescribeRequest, DetectPoseRequest, EditRequest, EnhancePromptRequest, GenerateRequest, MusicRequest,\n                     RemoveBackgroundRequest, SpeechRequest, TranscribeRequest, UpscaleRequest, VideoRequest, decode_audio_data,\n                     decode_image_data)\n\nTERMINAL = {"succeeded", "failed", "cancelled"}\ndef _diffusers_runner():\n    from .diffusers_runner import DiffusersQwenRunner\n    return DiffusersQwenRunner()\n\n\nRUNNERS = {"qwen": ResidentQwenRunner, "qwen-cli": QwenRunner, "diffusers": _diffusers_runner, "demo": DemoRunner}\n\n\ndef default_video_runner(runner):\n    """Demo video for the demo engine; LTX video only when the notebook configured its weights."""\n    if runner.mode == "demo":\n        return DemoVideoRunner()\n    return ResidentVideoRunner() if os.environ.get("ZERO_VIDEO_DIFFUSION") else None\n\n\ndef default_helpers(runner):\n    """Audio and language helpers the notebook configured (all of them, as demos, for the demo engine)."""\n    if runner.mode == "demo":\n        return DemoPromptHelper(), DemoTranscriber(), DemoSpeaker(), DemoMusicRunner()\n    env = os.environ.get\n    return (PromptHelper() if env("ZERO_VL") or env("DEEPSEEK_API_KEY") else None,\n            Transcriber() if env("ZERO_WHISPER") else None,\n            Speaker() if env("ZERO_TTS") else None,\n            MusicRunner() if env("ZERO_MUSIC") else None)\n\n\ndef default_vision(runner):\n    """Upscaler and pose detector the notebook configured (demos for the demo engine)."""\n    if runner.mode == "demo":\n        return DemoUpscaler(), DemoPoseDetector()\n    return (Upscaler() if os.environ.get("ZERO_UPSCALER") else None,\n            PoseDetector() if os.environ.get("ZERO_POSE_MODEL") else None)\n\n\nAUDIO_KINDS = ("speech", "music")\nDIRECT_KINDS = ("speech", "upscale")  # served by a helper, not by a swappable engine\n\n\ndef create_app(runner=None, token=None, output_dir=None, matter="auto", video_runner="auto", engine_swap=None,\n               helpers="auto", vision="auto"):\n    choice = os.environ.get("ZERO_RUNNER", "qwen")\n    if runner is None and choice not in RUNNERS:\n        raise RuntimeError("ZERO_RUNNER must be one of: " + ", ".join(RUNNERS))\n    runner = runner or RUNNERS[choice]()\n    video_runner = default_video_runner(runner) if video_runner == "auto" else video_runner\n    assistant, transcriber, speaker, music_runner = default_helpers(runner) if helpers == "auto" else helpers\n    upscaler, pose_detector = default_vision(runner) if vision == "auto" else vision\n    engines = EngineSet(runner, video_runner, swap=engine_swap, music=music_runner)\n    # Background removal is optional and separate from the image model; None disables it.\n    matter = default_matter(runner.mode) if matter == "auto" else matter\n    token = token or os.environ.get("ZERO_API_TOKEN", "")\n    if len(token) < 24:\n        raise RuntimeError("Set ZERO_API_TOKEN to a random secret of at least 24 characters.")\n    outputs = Path(output_dir or os.environ.get("ZERO_OUTPUT_DIR", "outputs")).resolve()\n    outputs.mkdir(parents=True, exist_ok=True)\n    task_dir = outputs / "tasks"\n    task_dir.mkdir(exist_ok=True)\n    for stale in [*task_dir.glob("*.png"), *task_dir.glob("*.json")]:\n        stale.unlink(missing_ok=True)  # tasks are ephemeral; results are fetched once by the studio\n    jobs, cancellation = {}, {}\n    tasks, task_flags = {}, {}\n    lock = RLock()\n    executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="studio-zero")\n    # Recover completed outputs; interrupted jobs are explicitly marked failed.\n    for path in outputs.glob("*.json"):\n        try:\n            job = json.loads(path.read_text(encoding="utf-8"))\n            if str(uuid.UUID(job["id"])) != job["id"]:\n                continue\n            if job["status"] not in TERMINAL:\n                job.update(status="failed", message="The service restarted before this generation finished.", progress=None)\n            jobs[job["id"]] = job\n        except (ValueError, KeyError, TypeError):\n            continue\n\n    @asynccontextmanager\n    async def lifespan(app):\n        # A resident engine begins loading its weights in the background; the API is usable at once.\n        engines.start()\n        if matter is not None and hasattr(matter, "start"):\n            matter.start()\n        yield\n        with lock:\n            for flag in [*cancellation.values(), *task_flags.values()]:\n                flag.set()\n        executor.shutdown(wait=True, cancel_futures=True)\n        engines.stop()\n        if matter is not None and hasattr(matter, "stop"):\n            matter.stop()\n\n    app = FastAPI(title="Studio Zero private engine", lifespan=lifespan, docs_url=None, redoc_url=None, openapi_url=None)\n    origins = [x.strip() for x in os.environ.get("ZERO_ALLOWED_ORIGINS", "http://localhost:5173,http://127.0.0.1:5173,http://localhost:4173,http://127.0.0.1:4173").split(",") if x.strip()]\n    if "*" in origins:\n        raise RuntimeError("Use exact Studio Zero origins, not a wildcard.")\n\n    @app.middleware("http")\n    async def body_limit(request: Request, call_next):\n        if request.method == "POST":\n            # Authenticate before buffering image-bearing requests.\n            if not hmac.compare_digest(request.headers.get("authorization", "").encode(), f"Bearer {token}".encode()):\n                from fastapi.responses import JSONResponse\n                return JSONResponse({"detail": "A valid access key is required."}, status_code=401)\n            large = ("/generate", "/remove-background", "/edit", "/video", "/describe", "/transcribe", "/speech", "/upscale",\n                     "/detect-pose")\n            limit = 48 * 1024 * 1024 if request.url.path in large else 128 * 1024\n            size = 0\n            chunks = []\n            async for chunk in request.stream():\n                size += len(chunk)\n                if size > limit:\n                    from fastapi.responses import JSONResponse\n                    return JSONResponse({"detail": "Request too large"}, status_code=413)\n                chunks.append(chunk)\n            request._body = b"".join(chunks)\n        return await call_next(request)\n\n    # Wrap body/auth rejections too, so the browser can read their error messages.\n    app.add_middleware(CORSMiddleware, allow_origins=origins, allow_methods=["GET", "POST"], allow_headers=["Authorization", "Content-Type"], allow_credentials=False)\n\n    def authorized(authorization: str = Header(default="")):\n        if not hmac.compare_digest(authorization.encode(), f"Bearer {token}".encode()):\n            raise HTTPException(401, "A valid access key is required.")\n\n    auth = [Depends(authorized)]\n\n    def persist(job):\n        tmp = outputs / f"{job[\'id\']}.json.tmp"\n        tmp.write_text(json.dumps(job, indent=2), encoding="utf-8")\n        tmp.replace(outputs / f"{job[\'id\']}.json")\n\n    def find(id):\n        if id not in jobs:\n            raise HTTPException(404, "Job not found.")\n        return jobs[id]\n\n    def snapshot(job):\n        return json.loads(json.dumps(job))\n\n    def output_path(id, kind, fmt=None):\n        if kind in AUDIO_KINDS:\n            return outputs / f"{id}.{fmt or \'wav\'}"\n        return outputs / f"{id}.{(fmt or \'mp4\') if kind == \'video\' else \'png\'}"\n\n    def run(id, kind, request):\n        flag = cancellation[id]\n        output = output_path(id, kind, jobs[id]["metadata"].get("format"))\n        def progress(value, message):\n            with lock:\n                if jobs[id]["status"] not in TERMINAL:\n                    jobs[id].update(progress=value, message=message)\n        working = {"video": "Creating your video…", "edit": "Editing your image…", "speech": "Recording the voice…",\n                   "music": "Composing your song…", "upscale": "Upscaling…"}.get(kind, "Creating your image…")\n        with lock:\n            if flag.is_set():\n                cancellation.pop(id, None)\n                return\n            jobs[id].update(status="running", message=working, progress=None)\n            persist(jobs[id])\n        try:\n            if kind == "speech":\n                engine = speaker\n            elif kind == "upscale":\n                engine = upscaler\n            else:\n                engine = engines.acquire({"video": "video", "music": "music"}.get(kind, "image"), flag, progress)\n            progress(None, working)\n            if kind in AUDIO_KINDS:\n                raw = output.with_suffix(".raw.wav")\n                if kind == "speech":\n                    voice = decode_audio_data(request.voice) if request.voice else None\n                    engine.speak(request, voice, raw, flag, progress)\n                else:\n                    engine.compose(request, raw, flag, progress)\n                if flag.is_set():\n                    raise Cancelled()\n                finish_audio(raw, output, jobs[id]["metadata"].get("format", "wav"))\n            elif kind == "upscale":\n                engine.upscale(request, decode_image_data(request.image), output, flag, progress)\n            elif kind == "video":\n                engine.generate_video(request, output, flag, progress)\n            elif kind == "edit":\n                engine.edit(request, output, flag, progress)\n            else:\n                engine.generate(request, output, flag, progress)\n            if flag.is_set():\n                raise Cancelled()\n            if kind in AUDIO_KINDS:\n                if not output.is_file() or output.stat().st_size <= 44:\n                    raise RuntimeError("The audio engine finished without producing sound.")\n            elif kind == "video":\n                if not output.is_file() or output.stat().st_size < 64:\n                    raise RuntimeError("The video engine finished without producing a video.")\n            else:\n                # Validate and embed the same metadata carried by the API and project file.\n                with Image.open(output) as image:\n                    image.load()\n                    expected = (jobs[id]["metadata"]["width"], jobs[id]["metadata"]["height"])\n                    if image.size != expected:\n                        raise RuntimeError("The engine returned unexpected image dimensions.")\n                    info = PngImagePlugin.PngInfo()\n                    info.add_text("studio_zero", json.dumps(jobs[id]["metadata"]))\n                    image.save(output.with_suffix(".tmp.png"), pnginfo=info)\n                output.with_suffix(".tmp.png").replace(output)\n            with lock:\n                if flag.is_set():\n                    raise Cancelled()\n                ready = {"video": "Your video is ready.", "edit": "Your edit is ready.", "speech": "Your voice clip is ready.",\n                         "music": "Your song is ready.", "upscale": "Your upscaled image is ready."}.get(kind, "Your image is ready.")\n                jobs[id].update(status="succeeded", progress=100, message=ready, output_id=id)\n        except Cancelled:\n            with lock:\n                jobs[id].update(status="cancelled", progress=None, message="Generation cancelled.")\n            output.unlink(missing_ok=True)\n            output.with_suffix(".raw.wav").unlink(missing_ok=True)\n        except Exception as exc:\n            with lock:\n                jobs[id].update(status="failed", progress=None, message=str(exc) if isinstance(exc, RuntimeError) else "The engine could not finish this job. Check the private service logs.")\n        finally:\n            with lock:\n                persist(jobs[id])\n                cancellation.pop(id, None)\n\n    def capabilities():\n        caps = ["text-to-image"]\n        if getattr(runner, "supports_references", False):\n            caps.append("reference")\n        if getattr(runner, "supports_transparency", False):\n            caps.append("transparent")\n        if getattr(runner, "supports_edit", False):\n            caps += ["inpaint", "outpaint", "image-to-image", "variations"]\n            if getattr(runner, "supports_references", False) or runner.mode == "demo":\n                caps.append("edit")\n        if engines.has("video"):\n            caps.append("video")\n        if matter is not None:\n            caps.append("remove-background")\n        if assistant is not None:\n            if not assistant.missing():\n                caps += ["describe", "enhance-prompt"]\n            elif getattr(assistant, "api_key", ""):\n                caps.append("enhance-prompt")\n        for name, helper in (("transcribe", transcriber), ("speech", speaker)):\n            if helper is not None and not helper.missing():\n                caps.append(name)\n        if engines.has("music"):\n            caps.append("music")\n        for name, helper in (("upscale", upscaler), ("detect-pose", pose_detector)):\n            if helper is not None and not helper.missing():\n                caps.append(name)\n        return caps\n\n    @app.get("/health", dependencies=auth)\n    def health():\n        ready, message = engines.readiness("image")\n        state = engines.state("image")\n        if hasattr(runner, "offload") and not (hasattr(runner, "missing") and runner.missing()):\n            state = {**state, "offload": runner.offload}\n        caps = capabilities()\n        if "video" in caps:\n            video_state = engines.state("video")\n            state = {**state, "video_engine": video_state["engine"], "video_loaded_in_seconds": video_state["loaded_in_seconds"],\n                     "video_model": engines.runner("video").model, "engine_swap": engines.swap}\n        if matter is not None:\n            matting_state = matter.state() if hasattr(matter, "state") else {"matting": "ready"}\n            state = {**state, **matting_state, "matting_mode": matter.mode}\n        models = {}\n        if assistant is not None:\n            models.update(assistant=assistant.model, prompt_rewriter=assistant.rewriter)\n        for name, helper in (("transcribe", transcriber), ("speech", speaker), ("music", music_runner),\n                             ("upscale", upscaler), ("pose", pose_detector)):\n            if helper is not None:\n                models[name] = helper.model\n        if "music" in caps:\n            state = {**state, "music_engine": engines.state("music")["engine"]}\n        if models:\n            state = {**state, "helper_models": models}\n        return {"ready": ready, "mode": runner.mode, "message": message, "capabilities": caps, "api_version": 1, **state}\n\n    def enqueue(kind, request, metadata):\n        if kind in DIRECT_KINDS:\n            engine, (ready, message) = (speaker if kind == "speech" else upscaler), (True, "")\n        else:\n            engine_kind = {"video": "video", "music": "music"}.get(kind, "image")\n            ready, message = engines.readiness(engine_kind)\n        if not ready:\n            raise HTTPException(503, message)\n        with lock:\n            if sum(j["status"] not in TERMINAL for j in jobs.values()) >= 4:\n                raise HTTPException(429, "Your engine is busy. Wait for a job to finish.")\n            id = str(uuid.uuid4())\n            if getattr(request, "seed", None) == -1:\n                request = request.model_copy(update={"seed": secrets.randbelow(2147483648)})\n            if kind not in DIRECT_KINDS:\n                engine = engines.runner(engine_kind)\n            metadata = {**metadata, "seed": getattr(request, "seed", None), "kind": kind, "model": engine.model, "jobId": id,\n                        "createdAt": datetime.now(timezone.utc).isoformat(), "demo": runner.mode == "demo",\n                        "runner": "procedural" if runner.mode == "demo" else getattr(engine, "runner_name", "diffusers" if kind in AUDIO_KINDS else "stable-diffusion.cpp"),\n                        "runnerRevision": getattr(engine, "runner_revision", None) or os.environ.get("ZERO_SD_REVISION", SD_REVISION)}\n            jobs[id] = {"id": id, "status": "queued", "progress": 0, "message": "Waiting for your engine…", "metadata": metadata}\n            cancellation[id] = Event()\n            persist(jobs[id])\n            executor.submit(run, id, kind, request)\n            return snapshot(jobs[id])\n\n    @app.post("/generate", dependencies=auth, status_code=202)\n    def generate(request: GenerateRequest):\n        if request.reference_images and not getattr(runner, "supports_references", False):\n            raise HTTPException(422, "This engine needs the vision encoder for reference images. Update the Colab notebook and restart the API.")\n        if request.transparent and "transparent" not in capabilities():\n            raise HTTPException(422, "This engine cannot create transparent images.")\n        metadata = {**request.model_dump(exclude={"reference_images"}), "reference_count": len(request.reference_images)}\n        return enqueue("image", request, metadata)\n\n    @app.post("/edit", dependencies=auth, status_code=202)\n    def edit(request: EditRequest):\n        caps = capabilities()\n        if request.operation not in caps:\n            extra = " Instruction edits need the vision encoder (ENABLE_REFERENCES in the notebook)." if request.operation == "edit" else ""\n            raise HTTPException(422, f"This engine does not offer {request.operation}.{extra}")\n        if request.reference_images and "reference" not in caps:\n            raise HTTPException(422, "This engine needs the vision encoder for reference images.")\n        if request.transparent and "transparent" not in caps:\n            raise HTTPException(422, "This engine cannot create transparent images.")\n        metadata = {**request.model_dump(exclude={"reference_images", "image", "mask", "strength"}),\n                    "reference_count": len(request.reference_images), "strength": request.effective_strength}\n        return enqueue("edit", request, metadata)\n\n    @app.post("/video", dependencies=auth, status_code=202)\n    def video(request: VideoRequest):\n        if "video" not in capabilities():\n            raise HTTPException(501, "This engine has no video model. Turn on ENABLE_VIDEO in the Colab notebook and restart the API.")\n        metadata = {**request.model_dump(exclude={"image", "end_image"}), "start_image": request.image is not None,\n                    "end_image": request.end_image is not None, "format": getattr(engines.runner("video"), "format", "mp4")}\n        return enqueue("video", request, metadata)\n\n    @app.get("/jobs", dependencies=auth)\n    def history():\n        with lock:\n            return [snapshot(j) for j in sorted(jobs.values(), key=lambda j: j[\'metadata\'][\'createdAt\'], reverse=True)[:50]]\n\n    @app.get("/jobs/{id}", dependencies=auth)\n    def job(id: str):\n        with lock:\n            return snapshot(find(id))\n\n    @app.post("/jobs/{id}/cancel", dependencies=auth)\n    def cancel(id: str):\n        with lock:\n            j = find(id)\n            if j["status"] not in TERMINAL:\n                cancellation[id].set()\n                j.update(status="cancelled", progress=None, message="Generation cancelled.")\n                persist(j)\n            return snapshot(j)\n\n    @app.get("/outputs/{id}", dependencies=auth)\n    def output(id: str):\n        with lock:\n            j = find(id)\n            if j["status"] != "succeeded":\n                raise HTTPException(409, "This output is not ready.")\n            kind, fmt = j["metadata"].get("kind", "image"), j["metadata"].get("format", "mp4")\n            path = output_path(id, kind, fmt)\n            if not path.is_file():\n                raise HTTPException(404, "Output no longer exists on this engine.")\n        media = ("audio/mpeg" if fmt == "mp3" else "audio/wav") if kind in AUDIO_KINDS else f"video/{fmt}" if kind == "video" else "image/png"\n        return FileResponse(path, media_type=media, headers={"Cache-Control": "private, no-store"})\n\n    @app.post("/speech", dependencies=auth, status_code=202)\n    def speech(request: SpeechRequest):\n        if "speech" not in capabilities():\n            raise HTTPException(501, "This engine has no voice model. Turn on ENABLE_VOICE in the Colab notebook and restart the API.")\n        metadata = {**request.model_dump(exclude={"voice"}), "cloned_voice": request.voice is not None, "format": audio_format()}\n        return enqueue("speech", request, metadata)\n\n    @app.post("/music", dependencies=auth, status_code=202)\n    def music(request: MusicRequest):\n        if "music" not in capabilities():\n            raise HTTPException(501, "This engine has no music model. Turn on ENABLE_MUSIC in the Colab notebook and restart the API.")\n        return enqueue("music", request, {**request.model_dump(), "format": audio_format()})\n\n    @app.post("/upscale", dependencies=auth, status_code=202)\n    def upscale(request: UpscaleRequest):\n        if "upscale" not in capabilities():\n            raise HTTPException(501, "This engine has no upscaling model. Turn on ENABLE_UPSCALE in the Colab notebook and restart the API.")\n        width, height = request.source_size\n        try:\n            check_upscale_size(width, height, request.scale)\n        except ValueError as exc:\n            raise HTTPException(422, str(exc))\n        metadata = {"scale": request.scale, "source_width": width, "source_height": height,\n                    "width": width * request.scale, "height": height * request.scale}\n        return enqueue("upscale", request, metadata)\n\n    # Short tasks: background removal, transcription, prompts ----------------------------\n    # Ephemeral (not persisted, not in /jobs history); they share the single GPU worker.\n    def run_task(id, work):\n        flag = task_flags[id]\n        with lock:\n            if flag.is_set():\n                task_flags.pop(id, None)\n                return\n            tasks[id].update(status="running", message=tasks[id]["working"])\n\n        def progress(value, message):\n            with lock:\n                if tasks[id]["status"] not in TERMINAL:\n                    tasks[id].update(progress=value, message=message)\n        try:\n            result = work(flag, progress, id)\n            if flag.is_set():\n                raise Cancelled()\n            with lock:\n                if flag.is_set():\n                    raise Cancelled()\n                tasks[id].update(status="succeeded", message=tasks[id]["done"], progress=100,\n                                 **({"result": result} if result is not None else {}))\n        except Cancelled:\n            with lock:\n                tasks[id].update(status="cancelled", message="Cancelled.")\n            (task_dir / f"{id}.png").unlink(missing_ok=True)\n        except Exception as exc:\n            with lock:\n                tasks[id].update(status="failed", message=str(exc) if isinstance(exc, RuntimeError) else f"{tasks[id][\'kind\']} failed. Check the private service log.")\n        finally:\n            with lock:\n                task_flags.pop(id, None)\n\n    def find_task(id):\n        if id not in tasks:\n            raise HTTPException(404, "Task not found. The engine may have restarted; try again.")\n        return tasks[id]\n\n    def submit_task(kind, work, working, done, **info):\n        with lock:\n            if sum(t["status"] not in TERMINAL for t in tasks.values()) >= 4:\n                raise HTTPException(429, "The engine is busy with other tasks. Wait for one to finish.")\n            id = str(uuid.uuid4())\n            tasks[id] = {"id": id, "kind": kind, "status": "queued", "message": "Waiting for your engine…", "progress": None,\n                         "working": working, "done": done, "createdAt": datetime.now(timezone.utc).isoformat(), **info}\n            task_flags[id] = Event()\n            # Keep only recent tasks and their masks.\n            for old in sorted(tasks.values(), key=lambda t: t["createdAt"])[:-20]:\n                if old["status"] in TERMINAL:\n                    tasks.pop(old["id"], None)\n                    (task_dir / f"{old[\'id\']}.png").unlink(missing_ok=True)\n            executor.submit(run_task, id, work)\n            return public_task(tasks[id])\n\n    def public_task(t):\n        return {k: v for k, v in t.items() if k not in ("working", "done")}\n\n    @app.post("/remove-background", dependencies=auth, status_code=202)\n    def remove_background(request: RemoveBackgroundRequest):\n        if matter is None:\n            raise HTTPException(501, "This engine has no background removal. Update the Colab notebook and restart the API.")\n        ready, message = matter.readiness()\n        if not ready:\n            raise HTTPException(503, message)\n        width, height = decode_image_data(request.image).size\n\n        def work(flag, progress, id):\n            image = decode_image_data(request.image)\n            mask = matter.matte(image)\n            if flag.is_set():\n                raise Cancelled()\n            if mask.size != image.size:\n                raise RuntimeError("Background removal returned a mask of the wrong size.")\n            mask.convert("L").save(task_dir / f"{id}.png")\n        return submit_task("remove-background", work, "Finding the subject…", "Subject found.",\n                           width=width, height=height, model=matter.model, demo=matter.mode == "demo")\n\n    def need(name, helper, setting):\n        if name not in capabilities():\n            raise HTTPException(501, f"This engine does not offer {name}. Turn on {setting} in the Colab notebook and restart the API.")\n        return helper\n\n    @app.post("/transcribe", dependencies=auth, status_code=202)\n    def transcribe(request: TranscribeRequest):\n        helper = need("transcribe", transcriber, "ENABLE_TRANSCRIBE")\n        data = decode_audio_data(request.audio)\n        return submit_task("transcribe", lambda flag, progress, _id: helper.transcribe(data, request.language, flag, progress),\n                           "Transcribing…", "Transcript ready.", model=helper.model, demo=helper.mode == "demo")\n\n    @app.post("/describe", dependencies=auth, status_code=202)\n    def describe(request: DescribeRequest):\n        helper = need("describe", assistant, "ENABLE_ASSISTANT")\n        return submit_task("describe", lambda flag, progress, _id: helper.describe(decode_image_data(request.image), request.purpose),\n                           "Looking at the image…", "Description ready.", model=helper.model, demo=helper.mode == "demo")\n\n    @app.post("/detect-pose", dependencies=auth, status_code=202)\n    def detect_pose(request: DetectPoseRequest):\n        helper = need("detect-pose", pose_detector, "ENABLE_POSE")\n        return submit_task("detect-pose", lambda flag, progress, _id: helper.detect(decode_image_data(request.image)),\n                           "Finding people and their pose…", "Pose found.", model=helper.model, demo=helper.mode == "demo")\n\n    @app.post("/enhance-prompt", dependencies=auth, status_code=202)\n    def enhance_prompt(request: EnhancePromptRequest):\n        helper = need("enhance-prompt", assistant, "ENABLE_ASSISTANT (or add a DEEPSEEK_API_KEY secret)")\n        return submit_task("enhance-prompt", lambda flag, progress, _id: helper.enhance(request.prompt, request.target),\n                           "Improving the prompt…", "Prompt ready.", model=helper.rewriter, demo=helper.mode == "demo")\n\n    @app.get("/tasks/{id}", dependencies=auth)\n    def task(id: str):\n        with lock:\n            return public_task(find_task(id))\n\n    @app.post("/tasks/{id}/cancel", dependencies=auth)\n    def cancel_task(id: str):\n        with lock:\n            t = find_task(id)\n            if t["status"] not in TERMINAL:\n                task_flags[id].set()\n                t.update(status="cancelled", message="Cancelled.")\n            return public_task(t)\n\n    @app.get("/tasks/{id}/mask", dependencies=auth)\n    def task_mask(id: str):\n        with lock:\n            t = find_task(id)\n            if t["status"] != "succeeded":\n                raise HTTPException(409, "This mask is not ready.")\n            path = task_dir / f"{id}.png"\n            if not path.is_file():\n                raise HTTPException(404, "This mask no longer exists on the engine.")\n        return FileResponse(path, media_type="image/png", headers={"Cache-Control": "private, no-store"})\n\n    return app\n', 'backend/assistant.py': '"""Prompt helper: turn a short idea into a detailed prompt, and describe an image as a prompt.\n\nLocal model: Qwen3-VL-2B-Instruct (vision + text, Apache-2.0), loaded on first use.\nOptional: with DEEPSEEK_API_KEY set, prompt rewriting uses DeepSeek\'s API (V4 Flash family) instead;\nimage description always stays local. Nothing else leaves the engine.\n"""\nimport os\nimport re\nfrom pathlib import Path\nfrom threading import RLock\n\nfrom PIL import Image, ImageStat\n\nfrom .audio import free_torch, helpers_resident\n\nVL_MODEL = "Qwen/Qwen3-VL-2B-Instruct"\n\nREWRITE = {\n    "image": ("You write prompts for a text-to-image model. Expand the user\'s idea into one vivid, concrete paragraph "\n              "of 60 to 120 words: subject, setting, composition, lighting, colour palette, style or medium, and camera "\n              "or lens when it fits. Keep every detail the user gave and do not contradict it. Add no text or lettering "\n              "unless asked. Reply with the prompt only."),\n    "edit": ("You write instructions for an AI image editor. Rewrite the user\'s request as one or two clear sentences "\n             "that say exactly what to change and what must stay the same. Reply with the instruction only."),\n    "video": ("You write prompts for a text/image-to-video model that also generates sound. Expand the user\'s idea into "\n              "60 to 120 words describing the subject, the motion over time, camera movement, pacing, lighting, and the "\n              "ambient sound or dialogue. Reply with the prompt only."),\n    "music": ("You write descriptions for a song generator. Turn the user\'s idea into at most 60 words covering genre, "\n              "BPM, key, mood and how it develops, the vocals (gender, timbre, delivery) and the arrangement. "\n              "Reply with the description only."),\n}\n\nDESCRIBE = {\n    "image": ("Describe this image as a text-to-image prompt that would recreate it: subject, setting, composition, "\n              "lighting, colours, and style or medium. One paragraph, no preamble."),\n    "video": ("Write a short image-to-video prompt that brings this image to life: what moves and how, the camera "\n              "movement, and the ambient sound. Two or three sentences, no preamble."),\n    "caption": "Describe this image in one or two plain sentences.",\n}\n\n\ndef clean(text: str) -> str:\n    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()\n    text = re.sub(r"^(prompt|instruction|description|rewritten prompt)\\s*:\\s*", "", text, flags=re.I).strip()\n    if len(text) > 1 and text[0] == text[-1] and text[0] in "\\"\'“”":\n        text = text[1:-1].strip()\n    return text.strip("“”").strip()\n\n\nclass PromptHelper:\n    mode = "qwen"\n    label = "assistant"\n\n    def __init__(self, model_dir=None, api_key=None, client=None):\n        self.path = model_dir or os.environ.get("ZERO_VL", "")\n        self.model = os.environ.get("ZERO_VL_ID", VL_MODEL)\n        self.api_key = api_key if api_key is not None else os.environ.get("DEEPSEEK_API_KEY", "")\n        self.api_model = os.environ.get("ZERO_DEEPSEEK_MODEL", "deepseek-flash")\n        self.api_base = os.environ.get("ZERO_DEEPSEEK_BASE", "https://api.deepseek.com")\n        self.client = client\n        self.lock = RLock()\n        self.vl = None\n\n    def missing(self):\n        if self.path and (Path(self.path) / "config.json").is_file():\n            return []\n        return ["vision-language model"]\n\n    @property\n    def rewriter(self):\n        return f"DeepSeek API ({self.api_model})" if self.api_key else self.model\n\n    # Local model ---------------------------------------------------------------------\n    def _load(self):\n        import torch\n        from transformers import AutoProcessor, Qwen3VLForConditionalGeneration\n        model = Qwen3VLForConditionalGeneration.from_pretrained(\n            self.path, dtype=torch.bfloat16, device_map="cuda" if torch.cuda.is_available() else "cpu")\n        return model, AutoProcessor.from_pretrained(self.path)\n\n    def _local(self, system, content, max_new_tokens=320):\n        import torch\n        with self.lock:\n            if self.vl is None:\n                self.vl = self._load()\n            model, processor = self.vl\n            messages = [{"role": "system", "content": [{"type": "text", "text": system}]},\n                        {"role": "user", "content": content}]\n            inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,\n                                                   return_dict=True, return_tensors="pt").to(model.device)\n            with torch.inference_mode():\n                ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7,\n                                     top_p=0.8, top_k=20)\n            text = processor.batch_decode(ids[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0]\n            if not helpers_resident():\n                self.vl = None\n                free_torch()\n        return clean(text)\n\n    def _api(self, system, prompt):\n        import httpx\n        client = self.client or httpx.Client(timeout=90)\n        response = client.post(f"{self.api_base}/chat/completions",\n                               headers={"Authorization": f"Bearer {self.api_key}"},\n                               json={"model": self.api_model, "temperature": 0.7, "max_tokens": 600,\n                                     "thinking": {"type": "disabled"},\n                                     "messages": [{"role": "system", "content": system},\n                                                  {"role": "user", "content": prompt}]})\n        if response.status_code >= 400:\n            raise RuntimeError(f"DeepSeek API returned {response.status_code}. Check the DEEPSEEK_API_KEY secret.")\n        return clean(response.json()["choices"][0]["message"]["content"])\n\n    # Public --------------------------------------------------------------------------\n    def enhance(self, prompt: str, target: str) -> dict:\n        system = REWRITE[target]\n        if self.api_key:\n            try:\n                return {"prompt": self._api(system, prompt), "model": self.rewriter}\n            except Exception as exc:\n                if self.missing():\n                    raise RuntimeError(str(exc)) from None\n        if self.missing():\n            raise RuntimeError("The prompt helper needs its model. Turn on ENABLE_ASSISTANT in the notebook.")\n        return {"prompt": self._local(system, [{"type": "text", "text": prompt}]), "model": self.model}\n\n    def describe(self, image: Image.Image, purpose: str) -> dict:\n        if self.missing():\n            raise RuntimeError("Describing images needs the vision model. Turn on ENABLE_ASSISTANT in the notebook.")\n        image = image.convert("RGB")\n        image.thumbnail((1024, 1024))\n        text = self._local(DESCRIBE[purpose], [{"type": "image", "image": image},\n                                               {"type": "text", "text": DESCRIBE[purpose]}])\n        return {"prompt": text, "model": self.model}\n\n\nclass DemoPromptHelper:\n    mode = "demo"\n    label = "assistant"\n    model = "studio-zero/demo-assistant"\n    rewriter = model\n\n    def missing(self):\n        return []\n\n    def enhance(self, prompt: str, target: str) -> dict:\n        extra = {"image": "detailed, soft natural light, balanced composition",\n                 "edit": "keep everything else unchanged",\n                 "video": "slow camera push-in, gentle ambient sound",\n                 "music": "mid-tempo, warm vocals, building chorus"}[target]\n        return {"prompt": f"{prompt.strip().rstrip(\'.\')}, {extra} (DEMO - NOT AI)", "model": self.model}\n\n    def describe(self, image: Image.Image, purpose: str) -> dict:\n        r, g, b = [int(v) for v in ImageStat.Stat(image.convert("RGB")).mean]\n        tone = max((("red", r), ("green", g), ("blue", b)), key=lambda x: x[1])[0]\n        light = "bright" if (r + g + b) / 3 > 140 else "dark"\n        return {"prompt": f"A {light} {image.width}x{image.height} image with mostly {tone} tones (DEMO DESCRIPTION - NOT AI)",\n                "model": self.model}\n', 'backend/audio.py': '"""Speech-to-text, voice cloning and music.\n\nTranscriber   faster-whisper (CTranslate2). GPU when the CUDA 12 libraries load, otherwise CPU int8.\nSpeaker       Chatterbox Turbo (English, voice cloning from a reference clip over 5 seconds, 24 kHz).\nMusicRunner   MiniMax Music 3 (diffusers modular pipeline): songs with vocals up to five minutes.\n\nThe small helpers load on first use and stay loaded when the GPU is large; the music model is a\nresident engine managed by EngineSet like the image and video engines. Each has a Demo twin that\nneeds no weights, so the studio and its tests run anywhere.\n"""\nimport ctypes\nimport glob\nimport io\nimport json\nimport math\nimport os\nimport re\nimport secrets\nimport shutil\nimport site\nimport struct\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport wave\nfrom pathlib import Path\nfrom threading import RLock, Thread\n\nfrom .runner import Cancelled, find_ffmpeg, gpu_memory_gib\n\nMUSIC_MODEL = "MiniMaxAI/MiniMax-Music3"\nTTS_MODEL = "ResembleAI/chatterbox-turbo"\nWHISPER_MODEL = "Systran/faster-whisper-base"\n\n\n# Audio files ------------------------------------------------------------------------------\ndef to_wav(data: bytes, output: Path, rate=16000, channels=1, max_seconds=None):\n    """Decode any audio (or a video\'s sound track) to PCM WAV with ffmpeg."""\n    ffmpeg = find_ffmpeg()\n    if not ffmpeg:\n        raise RuntimeError("ffmpeg is needed to read audio. Install it on the engine host.")\n    with tempfile.NamedTemporaryFile(suffix=".bin", delete=False) as source:\n        source.write(data)\n    try:\n        command = [ffmpeg, "-v", "error", "-y", "-i", source.name, "-vn", "-ac", str(channels), "-ar", str(rate)]\n        if max_seconds:\n            command += ["-t", str(max_seconds)]\n        result = subprocess.run(command + ["-c:a", "pcm_s16le", str(output)], capture_output=True, text=True, timeout=600)\n        if result.returncode or not output.is_file() or output.stat().st_size <= 44:\n            raise RuntimeError("Could not read this audio. Use WAV, MP3, M4A, OGG, WebM, FLAC or a video with sound.")\n    finally:\n        os.unlink(source.name)\n    return output\n\n\ndef audio_format():\n    """mp3 keeps projects small; wav when ffmpeg is missing or ZERO_AUDIO_FORMAT=wav."""\n    wanted = os.environ.get("ZERO_AUDIO_FORMAT", "mp3").lower()\n    return "mp3" if wanted == "mp3" and find_ffmpeg() else "wav"\n\n\ndef finish_audio(raw: Path, output: Path, fmt: str):\n    """Move the engine\'s WAV into place, encoding it to MP3 when asked."""\n    if fmt == "wav":\n        raw.replace(output)\n        return\n    result = subprocess.run([find_ffmpeg(), "-v", "error", "-y", "-i", str(raw), "-c:a", "libmp3lame", "-b:a", "192k", str(output)],\n                            capture_output=True, text=True, timeout=600)\n    raw.unlink(missing_ok=True)\n    if result.returncode or not output.is_file():\n        raise RuntimeError("Could not encode the audio as MP3.")\n\n\ndef read_pcm(path: Path):\n    """A 16-bit mono WAV as float32 samples in [-1, 1]."""\n    import numpy as np\n    with wave.open(str(path), "rb") as w:\n        return np.frombuffer(w.readframes(w.getnframes()), dtype="<i2").astype(np.float32) / 32768\n\n\ndef wav_seconds(path: Path) -> float:\n    with wave.open(str(path), "rb") as w:\n        return w.getnframes() / float(w.getframerate())\n\n\ndef write_wav(path: Path, samples, rate: int):\n    """Write float samples in [-1, 1]: a list (mono) or a list of per-channel lists."""\n    channels = samples if samples and isinstance(samples[0], (list, tuple)) else [samples]\n    frames = bytearray()\n    for i in range(len(channels[0])):\n        for channel in channels:\n            frames += struct.pack("<h", int(max(-1.0, min(1.0, channel[i])) * 32767))\n    with wave.open(str(path), "wb") as w:\n        w.setnchannels(len(channels)); w.setsampwidth(2); w.setframerate(rate)\n        w.writeframes(bytes(frames))\n\n\ndef write_array_wav(path: Path, audio, rate: int):\n    """Write a numpy array shaped (samples,) or (channels, samples) as 16-bit WAV."""\n    import numpy as np\n    audio = np.asarray(audio, dtype=np.float32)\n    if audio.ndim == 1:\n        audio = audio[None, :]\n    if audio.shape[0] > audio.shape[-1]:  # (samples, channels)\n        audio = audio.T\n    pcm = (np.clip(audio, -1, 1) * 32767).astype("<i2").T.copy()\n    with wave.open(str(path), "wb") as w:\n        w.setnchannels(pcm.shape[1]); w.setsampwidth(2); w.setframerate(int(rate))\n        w.writeframes(pcm.tobytes())\n\n\ndef srt(segments) -> str:\n    def stamp(t):\n        ms = int(round(t * 1000))\n        return f"{ms // 3600000:02d}:{ms // 60000 % 60:02d}:{ms // 1000 % 60:02d},{ms % 1000:03d}"\n    return "\\n".join(f"{i}\\n{stamp(s[\'start\'])} --> {stamp(s[\'end\'])}\\n{s[\'text\'].strip()}\\n"\n                     for i, s in enumerate(segments, 1))\n\n\ndef split_sentences(text: str, limit=280):\n    """Chunks under `limit` characters, cut at sentence ends where possible (TTS handles short inputs best)."""\n    parts, current = [], ""\n    for sentence in re.split(r"(?<=[.!?…])\\s+|\\n+", text.strip()):\n        sentence = sentence.strip()\n        while len(sentence) > limit:\n            cut = sentence.rfind(" ", 0, limit)\n            cut = cut if cut > limit // 2 else limit\n            if current:\n                parts.append(current); current = ""\n            parts.append(sentence[:cut].strip()); sentence = sentence[cut:].strip()\n        if not sentence:\n            continue\n        if current and len(current) + 1 + len(sentence) > limit:\n            parts.append(current); current = sentence\n        else:\n            current = f"{current} {sentence}".strip()\n    if current:\n        parts.append(current)\n    return parts\n\n\ndef helpers_resident():\n    """Keep small helper models loaded on big GPUs; unload after each use on small ones."""\n    mode = os.environ.get("ZERO_HELPERS_RESIDENT", "auto").lower()\n    if mode in ("1", "on", "true", "yes"):\n        return True\n    if mode in ("0", "off", "false", "no"):\n        return False\n    vram = gpu_memory_gib()\n    return vram is not None and vram >= 40\n\n\ndef free_torch():\n    if "torch" in sys.modules:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n\n_cuda12 = []\n\n\ndef preload_cuda12():\n    """CTranslate2 needs cuBLAS 12 and cuDNN 9; on Colab they come from pip NVIDIA packages torch installed."""\n    if _cuda12:\n        return _cuda12[0]\n    ok = False\n    try:\n        roots = [p for p in site.getsitepackages() + [site.getusersitepackages()] if p]\n        names = ["cublas/lib/libcublasLt.so.12", "cublas/lib/libcublas.so.12", "cudnn/lib/libcudnn.so.9"]\n        for name in names:\n            for root in roots:\n                hits = glob.glob(os.path.join(root, "nvidia", name))\n                if hits:\n                    ctypes.CDLL(hits[0], mode=ctypes.RTLD_GLOBAL)\n                    break\n        for extra in ("cudnn/lib/libcudnn_ops.so.9", "cudnn/lib/libcudnn_cnn.so.9"):\n            for root in roots:\n                hits = glob.glob(os.path.join(root, "nvidia", extra))\n                if hits:\n                    ctypes.CDLL(hits[0], mode=ctypes.RTLD_GLOBAL)\n                    break\n        ok = True\n    except OSError:\n        ok = False\n    _cuda12.append(ok)\n    return ok\n\n\n# Transcription ----------------------------------------------------------------------------\nclass Transcriber:\n    mode = "qwen"\n    label = "transcription"\n\n    def __init__(self, model_dir=None):\n        self.path = model_dir or os.environ.get("ZERO_WHISPER", "")\n        self.model = os.environ.get("ZERO_WHISPER_ID", WHISPER_MODEL)\n        self.lock = RLock()\n        self.whisper = None\n        self.device = None\n\n    def missing(self):\n        return [] if self.path and (Path(self.path) / "model.bin").is_file() else ["speech-to-text model"]\n\n    def _load(self):\n        from faster_whisper import WhisperModel\n        preference = os.environ.get("ZERO_WHISPER_DEVICE", "auto")\n        if preference != "cpu":\n            try:\n                preload_cuda12()\n                model = WhisperModel(self.path, device="cuda", compute_type="float16")\n                return model, "cuda"\n            except Exception:\n                if preference == "cuda":\n                    raise\n        return WhisperModel(self.path, device="cpu", compute_type="int8"), "cpu"\n\n    def transcribe(self, data: bytes, language, cancelled, progress=lambda *a: None):\n        with self.lock:\n            if self.whisper is None:\n                progress(None, "Loading the speech-to-text model…")\n                self.whisper, self.device = self._load()\n            with tempfile.TemporaryDirectory() as tmp:\n                audio = to_wav(data, Path(tmp) / "audio.wav")\n                total = max(wav_seconds(audio), 0.01)\n                # Pass samples, not a path: faster-whisper\'s own PyAV decoding breaks on current PyAV\n                # (it passes an option PyAV removed), and ffmpeg has already decoded the audio anyway.\n                segments, info = self.whisper.transcribe(read_pcm(audio), language=language, beam_size=5, vad_filter=True)\n                out = []\n                for segment in segments:  # a lazy generator: the work happens while iterating\n                    if cancelled.is_set():\n                        raise Cancelled()\n                    out.append({"start": round(segment.start, 2), "end": round(segment.end, 2), "text": segment.text.strip()})\n                    progress(min(99, round(segment.end * 100 / total)), "Transcribing…")\n            if not helpers_resident():\n                self.whisper = None\n                free_torch()\n        text = " ".join(s["text"] for s in out).strip()\n        return {"text": text, "language": info.language, "duration": round(total, 2), "segments": out,\n                "srt": srt(out), "model": self.model, "device": self.device}\n\n\nclass DemoTranscriber:\n    mode = "demo"\n    label = "transcription"\n    model = "studio-zero/demo-transcriber"\n    device = "cpu"\n\n    def missing(self):\n        return []\n\n    def transcribe(self, data: bytes, language, cancelled, progress=lambda *a: None):\n        try:\n            with wave.open(io.BytesIO(data), "rb") as w:\n                total = w.getnframes() / float(w.getframerate())\n        except (wave.Error, EOFError):\n            total = max(1.0, len(data) / 16000)\n        segments, t = [], 0.0\n        while t < total and len(segments) < 200:\n            end = min(total, t + 4)\n            segments.append({"start": round(t, 2), "end": round(end, 2), "text": f"DEMO TRANSCRIPT - NOT AI ({len(segments) + 1})"})\n            t = end\n        return {"text": " ".join(s["text"] for s in segments), "language": language or "en", "duration": round(total, 2),\n                "segments": segments, "srt": srt(segments), "model": self.model, "device": self.device}\n\n\n# Voice --------------------------------------------------------------------------------------\ndef float32_loudness(tts):\n    """Chatterbox expects NumPy < 2: there, loudness normalisation keeps float32 samples. Under NumPy 2\n    (Colab\'s Python 3.13) a float64 gain promotes the clip to float64 and voice cloning fails in the\n    speech tokenizer ("expected scalar type Float but found Double"). Cast the result back."""\n    import numpy as np\n    original = tts.norm_loudness\n    tts.norm_loudness = lambda wav, sr, target_lufs=-27: np.asarray(original(wav, sr, target_lufs), dtype=np.float32)\n    return tts\n\n\nclass Speaker:\n    mode = "qwen"\n    label = "voice"\n    rate = 24000\n\n    def __init__(self, model_dir=None):\n        self.path = model_dir or os.environ.get("ZERO_TTS", "")\n        self.model = os.environ.get("ZERO_TTS_ID", TTS_MODEL)\n        self.lock = RLock()\n        self.tts = None\n\n    def missing(self):\n        need = ("t3_turbo_v1.safetensors", "s3gen_meanflow.safetensors", "ve.safetensors", "conds.pt")\n        if not self.path or not all((Path(self.path) / f).is_file() for f in need):\n            return ["voice model"]\n        return []\n\n    def _load(self):\n        import torch\n        from chatterbox.tts_turbo import ChatterboxTurboTTS\n        tts = ChatterboxTurboTTS.from_local(self.path, "cuda" if torch.cuda.is_available() else "cpu")\n        return float32_loudness(tts)\n\n    def speak(self, request, voice: bytes | None, output: Path, cancelled, progress=lambda *a: None):\n        import numpy as np\n        import torch\n        with self.lock:\n            if self.tts is None:\n                progress(None, "Loading the voice model…")\n                self.tts = self._load()\n            with tempfile.TemporaryDirectory() as tmp:\n                reference = None\n                if voice is not None:\n                    reference = to_wav(voice, Path(tmp) / "voice.wav", rate=self.rate, max_seconds=30)\n                    if wav_seconds(reference) <= 5.0:\n                        raise RuntimeError("The voice sample must be longer than 5 seconds (about 10 seconds works best).")\n                seed = request.seed if request.seed >= 0 else secrets.randbelow(2147483647)\n                torch.manual_seed(seed)\n                chunks = split_sentences(request.text)\n                pieces = []\n                for i, chunk in enumerate(chunks):\n                    if cancelled.is_set():\n                        raise Cancelled()\n                    progress(round(i * 100 / len(chunks)), f"Speaking part {i + 1} of {len(chunks)}…")\n                    wav = self.tts.generate(chunk, audio_prompt_path=str(reference) if reference and i == 0 else None,\n                                            temperature=request.temperature)\n                    pieces.append(wav.squeeze(0).cpu().numpy())\n                    pieces.append(np.zeros(int(self.rate * 0.18), dtype=np.float32))\n                write_array_wav(output, np.concatenate(pieces[:-1]), self.tts.sr)\n            if not helpers_resident():\n                self.tts = None\n                free_torch()\n\n\nclass DemoSpeaker:\n    mode = "demo"\n    label = "voice"\n    model = "studio-zero/demo-voice"\n    rate = 24000\n\n    def missing(self):\n        return []\n\n    def speak(self, request, voice, output: Path, cancelled, progress=lambda *a: None):\n        """A short beep per word: clearly synthetic, shaped like speech timing."""\n        samples = []\n        words = request.text.split()[:400]\n        for i, word in enumerate(words):\n            if cancelled.is_set():\n                raise Cancelled()\n            freq = 330 + (hash(word) % 7) * 40\n            length = int(self.rate * min(0.45, 0.08 + 0.03 * len(word)))\n            samples += [0.25 * math.sin(2 * math.pi * freq * n / self.rate) * math.sin(math.pi * n / length) for n in range(length)]\n            samples += [0.0] * int(self.rate * 0.07)\n        write_wav(output, samples or [0.0] * self.rate, self.rate)\n\n\n# Music --------------------------------------------------------------------------------------\nclass MusicRunner:\n    """MiniMax Music 3 kept in memory between songs; swapped with the image/video engines when needed."""\n    mode = "qwen"\n    label = "music"\n    vram_gib = 24\n\n    def __init__(self, model_dir=None, loader=None):\n        self.path = model_dir or os.environ.get("ZERO_MUSIC", "")\n        self.model = os.environ.get("ZERO_MUSIC_ID", MUSIC_MODEL)\n        self.loader = loader or self._load\n        self.lock = RLock()\n        self.pipe = None\n        self.status, self.detail, self.loaded_in, self.last_start = "stopped", "", None, 0.0\n        self._generation = 0\n\n    def missing(self):\n        root = Path(self.path) if self.path else None\n        if not root or not (root / "modular_model_index.json").is_file():\n            return ["music model"]\n        gone = [d for d in ("language_model", "transformer", "rvq_depth_decoder", "vocoder", "condition_encoder", "tokenizer")\n                if not (root / d).is_dir()]\n        return [f"music model ({\', \'.join(gone)})"] if gone else []\n\n    def local_index(self):\n        """The published index names the Hub repo for every component; point it at this folder instead."""\n        index = Path(self.path) / "modular_model_index.json"\n        config = json.loads(index.read_text(encoding="utf-8"))\n        changed = False\n        for name, value in config.items():\n            if isinstance(value, list) and len(value) == 3 and isinstance(value[2], dict):\n                if value[2].get("pretrained_model_name_or_path") != str(self.path):\n                    value[2]["pretrained_model_name_or_path"] = str(self.path)\n                    changed = True\n        if changed:\n            text = json.dumps(config, indent=2)\n            if index.is_symlink():\n                index.unlink()\n            index.write_text(text, encoding="utf-8")\n        return config\n\n    def start(self):\n        with self.lock:\n            if self.status in ("loading", "ready") or self.missing():\n                return\n            self.status, self.detail, self.loaded_in = "loading", "", None\n            self.last_start = time.monotonic()\n            self._generation += 1\n            generation = self._generation\n        Thread(target=self._load_thread, args=(generation,), daemon=True, name="studio-zero-music").start()\n\n    def _load_thread(self, generation):\n        started = time.monotonic()\n        try:\n            pipe = self.loader()\n        except Exception as exc:\n            with self.lock:\n                if generation == self._generation:\n                    self.status, self.detail = "failed", f"The music model could not load: {type(exc).__name__}: {exc}"[:600]\n            return\n        with self.lock:\n            if generation != self._generation:\n                return\n            self.pipe = pipe\n            self.status, self.loaded_in = "ready", round(time.monotonic() - started, 1)\n\n    def _load(self):\n        import torch\n        from diffusers import ModularPipeline\n        self.local_index()\n        pipe = ModularPipeline.from_pretrained(self.path)\n        pipe.load_components(dtype=torch.bfloat16)\n        vram = gpu_memory_gib()\n        if vram is not None and vram < 30:\n            from diffusers.hooks.group_offloading import apply_group_offloading\n            apply_group_offloading(pipe.language_model, onload_device=torch.device("cuda"),\n                                   offload_type="leaf_level", use_stream=True)\n            for name in ("transformer", "rvq_depth_decoder", "vocoder", "condition_encoder"):\n                component = getattr(pipe, name, None)\n                if component is not None:\n                    component.to("cuda")\n        else:\n            pipe.to("cuda")\n        return pipe\n\n    def stop(self):\n        with self.lock:\n            self._generation += 1\n            pipe, self.pipe = self.pipe, None\n            self.status = "stopped"\n        if pipe is not None:\n            del pipe\n            free_torch()\n\n    def state(self):\n        with self.lock:\n            return {"engine": self.status, "loaded_in_seconds": self.loaded_in}\n\n    def readiness(self):\n        missing = self.missing()\n        if missing:\n            return False, "Configure missing " + ", ".join(missing)\n        status = self.state()["engine"]\n        if status == "stopped" or (status == "failed" and time.monotonic() - self.last_start > 60):\n            self.start()\n            status = self.state()["engine"]\n        if status == "ready":\n            return True, "Music model ready"\n        if status == "loading":\n            return False, "Loading the music model into memory…"\n        return False, self.detail or "The music model is not running."\n\n    def compose(self, request, output: Path, cancelled, progress):\n        import torch\n        ready, message = self.readiness()\n        if not ready:\n            raise RuntimeError(message)\n        pipe = self.pipe\n        seed = request.seed if request.seed >= 0 else secrets.randbelow(2147483647)\n        lyrics = "[instrumental]" if request.instrumental else request.lyrics.strip()\n        style = request.style.strip()\n        if request.instrumental and "instrumental" not in style.lower():\n            style += " Instrumental, no vocals."\n        done = []\n\n        def ticker():  # the autoregressive stage has no step callback; show elapsed time instead\n            started = time.monotonic()\n            while not done:\n                elapsed = int(time.monotonic() - started)\n                progress(None, f"Composing ({elapsed} s)… a {request.duration}-second song takes a while.")\n                time.sleep(2)\n        Thread(target=ticker, daemon=True).start()\n        try:\n            audio = pipe(prompt=style, lyrics=lyrics, audio_duration=float(request.duration),\n                         generator=torch.Generator("cuda").manual_seed(seed), output="audios")[0]\n        finally:\n            done.append(True)\n        if cancelled.is_set():\n            raise Cancelled()\n        if hasattr(audio, "detach"):\n            audio = audio.detach().float().cpu().numpy()\n        write_array_wav(output, audio, pipe.sampling_rate)\n\n\nclass DemoMusicRunner:\n    mode = "demo"\n    label = "music"\n    model = "studio-zero/demo-music"\n    rate = 22050\n\n    def missing(self):\n        return []\n\n    def readiness(self):\n        return True, "Demo music (synthesized) - not AI"\n\n    def compose(self, request, output: Path, cancelled, progress):\n        """A simple arpeggio in stereo, as long as requested (capped at 30 s for speed)."""\n        seconds = min(request.duration, 30)\n        notes = [261.63, 329.63, 392.0, 523.25, 392.0, 329.63]\n        left, right = [], []\n        step = int(self.rate * 0.25)\n        for i in range(int(seconds * 4)):\n            if cancelled.is_set():\n                raise Cancelled()\n            freq = notes[(i + (request.seed % 3 if request.seed > 0 else 0)) % len(notes)]\n            for n in range(step):\n                envelope = math.exp(-3 * n / step)\n                value = 0.2 * envelope * math.sin(2 * math.pi * freq * n / self.rate)\n                left.append(value); right.append(value * (0.6 + 0.4 * math.sin(i / 3)))\n            if i % 16 == 0:\n                progress(round(i * 100 / (seconds * 4)), "Composing (demo)…")\n        write_wav(output, [left, right], self.rate)\n', 'backend/diffusers_runner.py': '"""Fast image engine: Qwen-Image 2.1 in diffusers with the Fun-Acc 4-step adapter.\n\nWhy a second image engine: Alibaba\'s Fun-Acc speed-up is Parallel Decoding Distillation (PDD), which\nneeds its own loader and scheduler (`qwenimage21_pdd.py`, `lora_utils_pdd.py`, downloaded at run time\nfrom alibaba-pai/Qwen-Image-2.1-Fun-Acc-LoRAs) on top of the native diffusers pipeline — it is not a\nplain LoRA that stable-diffusion.cpp could apply. 4 steps instead of ~40.\n\nWeights: the uncensored UC GGUF (the same file the stable-diffusion.cpp engine uses) is dequantized to\nBF16 and loaded as the transformer; Qwen/Qwen-Image-2.1 supplies only the text encoder, VAE, processor\nand configs ("companion" directory). Without ZERO_DIFFUSION the companion\'s own transformer is used.\n\nEdits: the pipeline has no mask/strength inputs, so inpaint, outpaint and image-to-image blend the\nsource\'s latents back in at every step boundary (flow matching: x_σ = (1 − σ)·x₀ + σ·ε). Masked areas\n(white) are generated; the rest follows the source exactly. Image-to-image starts from the first step\nboundary at or below `strength`. Instruction edits condition on the source as reference image 1.\n\nAll torch calls go through `TorchOps`, so the scheduling logic is testable without torch.\n"""\nimport gc\nimport os\nimport secrets\nimport sys\nimport time\nfrom contextlib import nullcontext\nfrom pathlib import Path\nfrom threading import RLock, Thread\n\nimport numpy as np\nfrom PIL import Image\n\nfrom .runner import Cancelled, gpu_memory_gib, transparent_prompt\nfrom .schema import decode_image_data\n\nDIFFUSERS_REVISION = "9f1246971270c84dcbe71233edb7a519596a5d02"\nFAST_STEPS = 4\n\n\nclass TorchOps:\n    """The few tensor operations the runner needs, on torch."""\n\n    def __init__(self):\n        import torch\n        self.torch = torch\n\n    def inference(self):\n        return self.torch.inference_mode()\n\n    def generator(self, seed):\n        return self.torch.Generator("cpu").manual_seed(int(seed))\n\n    def randn_like(self, like, generator):\n        noise = self.torch.randn(tuple(like.shape), generator=generator, dtype=self.torch.float32)\n        return noise.to(device=like.device, dtype=like.dtype)\n\n    def vae_input(self, pipe, image, width, height):\n        tensor = pipe.image_processor.preprocess(image, width=width, height=height).unsqueeze(2)\n        return tensor.to(device=pipe._execution_device, dtype=pipe.vae.dtype)\n\n    def from_numpy(self, array, like):\n        return self.torch.from_numpy(np.ascontiguousarray(array)).to(device=like.device, dtype=like.dtype)\n\n    def free(self):\n        gc.collect()\n        if self.torch.cuda.is_available():\n            self.torch.cuda.empty_cache()\n\n\ndef latent_mask(mask: Image.Image, width: int, height: int):\n    """Engine mask (white = regenerate) → (1, width·height, 1) float array on the latent grid.\n    A latent cell regenerates if any pixel inside it is marked (like stable-diffusion.cpp\'s NearestMax)."""\n    grey = np.asarray(mask.convert("L"), dtype=np.float32) / 255.0\n    rows = np.array_split(np.arange(grey.shape[0]), height)\n    cols = np.array_split(np.arange(grey.shape[1]), width)\n    cells = np.zeros((height, width), dtype=np.float32)\n    for y, r in enumerate(rows):\n        block = grey[r[0]:r[-1] + 1]\n        for x, c in enumerate(cols):\n            cells[y, x] = 1.0 if block[:, c[0]:c[-1] + 1].max() > 0.5 else 0.0\n    return cells.reshape(1, height * width, 1)\n\n\ndef start_index(sigmas, strength):\n    """First step boundary at or below `strength` (at least one real step remains)."""\n    values = [float(s) for s in sigmas]\n    if strength >= 0.999:\n        return 0\n    for k, s in enumerate(values):\n        if s <= strength:\n            return max(1, min(k, len(values) - 2))\n    return len(values) - 2\n\n\nclass LatentBlend:\n    """Keeps the source\'s latents where the mask is 0, at the noise level of each step boundary."""\n\n    def __init__(self, x0, noise, mask=None, strength=1.0):\n        self.x0, self.noise, self.mask, self.strength = x0, noise, mask, strength\n        self.start = None\n\n    def known(self, sigma):\n        return self.x0 * (1 - sigma) + self.noise * sigma\n\n    def __call__(self, pipe, i, t, kwargs):\n        sigmas = pipe.scheduler.sigmas\n        if self.start is None:\n            self.start = start_index(sigmas, self.strength)\n        sigma = float(sigmas[i + 1])\n        latents = kwargs["latents"]\n        known = self.known(sigma)\n        if i + 1 <= self.start:\n            blended = known                       # still before the chosen starting noise level\n        elif self.mask is None:\n            return {}\n        else:\n            blended = self.mask * latents + (1 - self.mask) * known\n        return {"latents": blended}\n\n\nclass DiffusersQwenRunner:\n    mode = "qwen"\n    runner_name = "diffusers"\n    runner_revision = DIFFUSERS_REVISION\n    label = "image"\n    supports_edit = True\n    supports_transparency = True\n    supports_references = True\n\n    def __init__(self, loader=None, ops=None):\n        self.model = os.environ.get("ZERO_MODEL_ID", "Qwen-Image-2.1 (diffusers)")\n        self.gguf = os.environ.get("ZERO_DIFFUSION", "")\n        self.companion = os.environ.get("ZERO_DIFFUSERS_COMPANION", "")\n        self.fast_lora = os.environ.get("ZERO_FAST_LORA", "")\n        self.fast_code = os.environ.get("ZERO_FAST_CODE", "") or (str(Path(self.fast_lora).parent.parent) if self.fast_lora else "")\n        self.fast = bool(self.fast_lora)\n        self.load_timeout = int(os.environ.get("ZERO_ENGINE_LOAD_TIMEOUT", "1800"))\n        self.loader = loader or self._load\n        self._ops = ops\n        self.lock = RLock()\n        self.pipe = None\n        self.pdd = None\n        self.status, self.detail, self.loaded_in, self.last_start = "stopped", "", None, 0.0\n        self.placement = None\n        self._generation = 0\n\n    @property\n    def ops(self):\n        if self._ops is None:\n            self._ops = TorchOps()\n        return self._ops\n\n    @property\n    def ready_message(self):\n        return ("Fast Qwen engine ready — 4 steps per image, model kept in memory" if self.fast\n                else "Qwen engine (diffusers) ready — model kept in memory")\n\n    # Configuration -------------------------------------------------------------\n    def missing(self):\n        problems = []\n        if not self.companion or not (Path(self.companion) / "model_index.json").is_file():\n            problems.append("companion model folder (text encoder, VAE)")\n        if self.gguf and not Path(self.gguf).is_file():\n            problems.append("diffusion weights")\n        if self.fast:\n            if not Path(self.fast_lora).is_file() or not (Path(self.fast_lora).parent / "pdd_config.json").is_file():\n                problems.append("Fun-Acc 4-step adapter")\n            for name in ("qwenimage21_pdd.py", "lora_utils_pdd.py"):\n                if not (Path(self.fast_code) / name).is_file():\n                    problems.append(name)\n        return problems\n\n    def weight_bytes(self):\n        """Approximate resident size in BF16: text encoder + transformer + VAE."""\n        total = 0\n        root = Path(self.companion) if self.companion else None\n        for folder in ("text_encoder", "vae"):\n            if root and (root / folder).is_dir():\n                total += sum(f.stat().st_size for f in (root / folder).glob("*.safetensors"))\n        if self.gguf and Path(self.gguf).is_file():\n            total += 14.3e9  # the transformer is dequantized to BF16\n        elif root and (root / "transformer").is_dir():\n            total += sum(f.stat().st_size for f in (root / "transformer").glob("*.safetensors"))\n        return total\n\n    def choose_placement(self):\n        """cuda (all on the GPU) · model (one component at a time) · sequential (smallest GPUs)."""\n        mode = os.environ.get("ZERO_OFFLOAD", "auto").strip().lower()\n        if mode in ("0", "off", "false", "no"):\n            return "cuda"\n        vram = gpu_memory_gib()\n        if mode in ("1", "on", "true", "yes"):\n            return "model" if vram is None or vram >= 20 else "sequential"\n        if vram is None:\n            return "model"\n        headroom = float(os.environ.get("ZERO_VRAM_HEADROOM_GIB", "6"))\n        if self.weight_bytes() / 2**30 + headroom <= vram:\n            return "cuda"\n        return "model" if vram >= 20 else "sequential"\n\n    @property\n    def offload(self):\n        return self.placement not in (None, "cuda")\n\n    # Lifecycle ---------------------------------------------------------------------\n    def start(self):\n        with self.lock:\n            if self.status in ("loading", "ready") or self.missing():\n                return\n            self.status, self.detail, self.loaded_in = "loading", "", None\n            self.last_start = time.monotonic()\n            self._generation += 1\n            generation = self._generation\n        Thread(target=self._load_thread, args=(generation,), daemon=True, name="studio-zero-diffusers").start()\n\n    def _load_thread(self, generation):\n        started = time.monotonic()\n        try:\n            pipe, pdd = self.loader()\n        except Exception as exc:  # report, never crash the API\n            with self.lock:\n                if generation == self._generation:\n                    self.status, self.detail = "failed", f"The fast engine could not load: {type(exc).__name__}: {exc}"[:600]\n            return\n        with self.lock:\n            if generation != self._generation:  # stopped meanwhile\n                return\n            self.pipe, self.pdd = pipe, pdd\n            self.status, self.loaded_in = "ready", round(time.monotonic() - started, 1)\n\n    def _load(self):\n        import torch\n        from diffusers import QwenImage21Pipeline, QwenImage21Transformer2DModel\n        transformer = None\n        if self.gguf:\n            from accelerate import init_empty_weights\n            from diffusers.models.model_loading_utils import load_gguf_checkpoint\n            from diffusers.quantizers.gguf.utils import dequantize_gguf_tensor\n            weights = load_gguf_checkpoint(self.gguf)\n            for name in list(weights):\n                weights[name] = dequantize_gguf_tensor(weights[name]).to(torch.bfloat16)\n            config = QwenImage21Transformer2DModel.load_config(self.companion, subfolder="transformer")\n            with init_empty_weights():\n                transformer = QwenImage21Transformer2DModel.from_config(config)\n            transformer.load_state_dict(weights, strict=True, assign=True)\n            transformer.eval().requires_grad_(False)\n            del weights\n        kwargs = {"torch_dtype": torch.bfloat16}\n        if transformer is not None:\n            kwargs["transformer"] = transformer\n        pipe = QwenImage21Pipeline.from_pretrained(self.companion, **kwargs)\n        pdd = None\n        if self.fast:\n            if self.fast_code not in sys.path:\n                sys.path.insert(0, self.fast_code)\n            import qwenimage21_pdd as module\n            config = module.load_pdd_lora(pipe.transformer, self.fast_lora)\n            pipe.transformer.eval()\n            pipe.scheduler = module.QwenImage21PDDScheduler.from_config(pipe.scheduler.config)\n            pipe.scheduler.register_to_config(**config)\n            pdd = {"module": module, "config": config, "sigmas": torch.tensor(config["pdd_sigmas"], dtype=torch.float32)}\n        self.placement = self.choose_placement()\n        if self.placement == "cuda":\n            pipe.to("cuda")\n        elif self.placement == "model":\n            pipe.enable_model_cpu_offload()\n        else:\n            pipe.enable_sequential_cpu_offload()\n        return pipe, pdd\n\n    def stop(self):\n        with self.lock:\n            self._generation += 1\n            pipe, self.pipe, self.pdd = self.pipe, None, None\n            self.status = "stopped"\n        if pipe is not None:\n            del pipe\n            if self._ops is not None or "torch" in sys.modules:\n                self.ops.free()\n\n    def state(self):\n        with self.lock:\n            return {"engine": self.status, "loaded_in_seconds": self.loaded_in}\n\n    def readiness(self):\n        missing = self.missing()\n        if missing:\n            return False, "Configure missing " + ", ".join(missing)\n        status = self.state()["engine"]\n        if status == "stopped" or (status == "failed" and time.monotonic() - self.last_start > 60):\n            self.start()\n            status = self.state()["engine"]\n        if status == "ready":\n            return True, self.ready_message\n        if status == "loading":\n            return False, "Loading the fast image model into memory. This happens once per engine start."\n        return False, self.detail or "The fast image engine is not running."\n\n    # Generation ---------------------------------------------------------------------\n    def _encode(self, image, width, height, generator):\n        pipe = self.pipe\n        encoded = pipe._encode_vae_image(self.ops.vae_input(pipe, image, width, height), generator)\n        channels, h, w = encoded.shape[1], encoded.shape[3], encoded.shape[4]\n        return pipe._pack_latents(encoded, 1, channels, h, w), h, w\n\n    def _run(self, prompt, request, cancelled, progress, images=None, source=None, mask=None, strength=1.0):\n        ready, message = self.readiness()\n        if not ready:\n            raise RuntimeError(message)\n        pipe, pdd, ops = self.pipe, self.pdd, self.ops\n        seed = request.seed if request.seed >= 0 else secrets.randbelow(2147483647)\n        steps = FAST_STEPS if pdd else request.steps\n        callbacks = []\n        if pdd:\n            callbacks.append(pdd["module"].pdd_step_callback(pipe.transformer, pdd["sigmas"], pdd["config"]["pdd_block_size"]))\n        kwargs = {"prompt": prompt, "height": request.height, "width": request.width, "num_inference_steps": steps,\n                  "generator": ops.generator(seed), "output_type": "pil",\n                  "callback_on_step_end_tensor_inputs": ["latents"]}\n        if pdd:\n            kwargs.update(true_cfg_scale=1.0, use_kv_cache=False)\n        elif request.negative_prompt.strip() and request.guidance > 1:\n            kwargs.update(true_cfg_scale=float(request.guidance), negative_prompt=request.negative_prompt)\n        if images:\n            kwargs.update(image=images, output_resolution=1024)\n        with ops.inference():\n            if source is not None:\n                x0, h, w = self._encode(source, request.width, request.height, ops.generator(seed))\n                noise = ops.randn_like(x0, ops.generator(seed + 1))\n                blend_mask = ops.from_numpy(latent_mask(mask, w, h), x0) if mask is not None else None\n                callbacks.append(LatentBlend(x0, noise, blend_mask, strength))\n                kwargs["latents"] = noise\n            def on_step(p, i, t, values):\n                out = {}\n                for callback in callbacks:\n                    out.update(callback(p, i, t, {**values, **out}) or {})\n                progress(round((i + 1) * 100 / steps), f"Step {i + 1} of {steps}…")\n                if cancelled.is_set():\n                    p._interrupt = True\n                return out\n            kwargs["callback_on_step_end"] = on_step\n            image = pipe(**kwargs).images[0]\n        if cancelled.is_set():\n            raise Cancelled()\n        return image\n\n    @staticmethod\n    def _images(values):\n        return [decode_image_data(v).convert("RGBA") for v in values]\n\n    def generate(self, request, output, cancelled, progress):\n        prompt = transparent_prompt(request.prompt) if request.transparent else request.prompt\n        image = self._run(prompt, request, cancelled, progress, images=self._images(request.reference_images) or None)\n        image.save(output)\n\n    def edit(self, request, output, cancelled, progress):\n        prompt = transparent_prompt(request.prompt) if request.transparent else request.prompt\n        source = decode_image_data(request.image).convert("RGBA")\n        references = self._images(request.reference_images)\n        if request.operation == "edit":\n            image = self._run(prompt, request, cancelled, progress, images=[source, *references])\n        elif request.operation in ("inpaint", "outpaint"):\n            mask = decode_image_data(request.mask)\n            image = self._run(prompt, request, cancelled, progress, images=[source, *references], source=source, mask=mask)\n        else:\n            image = self._run(prompt, request, cancelled, progress, images=references or None, source=source,\n                              strength=request.effective_strength)\n        image.save(output)\n', 'backend/engines.py': '"""Which engine serves which kind of job, and swapping them on a single GPU.\n\nResident engines serve images, video or music. On one GPU (the default, ZERO_ENGINE_SWAP=1) a job\nfor another kind stops the loaded engines and starts the one it needs, inside the single worker\nthread, so jobs never race - unless the incoming engine declares its size (`vram_gib`) and fits in\nthe GPU memory that is free right now, in which case everything stays loaded.\nWith ZERO_ENGINE_SWAP=0 (enough memory for all) engines stay loaded side by side.\nDemo and command-line runners have no lifecycle and are used as they are.\n"""\nimport os\nimport subprocess\nimport time\nfrom threading import RLock\n\nfrom .runner import Cancelled\n\n\ndef _resident(runner):\n    return runner is not None and hasattr(runner, "start") and hasattr(runner, "state")\n\n\ndef gpu_free_gib():\n    try:\n        text = subprocess.run(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits"],\n                              capture_output=True, text=True, timeout=20, check=True).stdout\n        return float(text.split()[0]) / 1024\n    except (OSError, subprocess.SubprocessError, ValueError, IndexError):\n        return None\n\n\nclass EngineSet:\n    def __init__(self, image, video=None, swap=None, music=None, free_memory=gpu_free_gib):\n        self.runners = {"image": image, **({"video": video} if video is not None else {}),\n                        **({"music": music} if music is not None else {})}\n        self.free_memory = free_memory\n        self.swap = (os.environ.get("ZERO_ENGINE_SWAP", "1") != "0") if swap is None else swap\n        self.active = "image"\n        self.lock = RLock()\n\n    def runner(self, kind):\n        return self.runners.get(kind)\n\n    def has(self, kind):\n        runner = self.runner(kind)\n        if runner is None:\n            return False\n        return not (hasattr(runner, "missing") and runner.missing())\n\n    def start(self):\n        if _resident(self.runners["image"]):\n            self.runners["image"].start()\n\n    def stop(self):\n        for runner in self.runners.values():\n            if _resident(runner):\n                runner.stop()\n\n    def readiness(self, kind):\n        runner = self.runner(kind)\n        if runner is None:\n            return False, f"This engine has no {kind} model. Update the Colab notebook and restart the API."\n        with self.lock:\n            inactive = self.swap and self.active != kind\n        if _resident(runner) and inactive:\n            missing = runner.missing()\n            if missing:\n                return False, "Configure missing " + ", ".join(missing)\n            return True, f"The {kind} model loads when it is first needed (the other model is unloaded to make room)."\n        return runner.readiness()\n\n    def state(self, kind):\n        runner = self.runner(kind)\n        if not _resident(runner):\n            ready = runner is not None and runner.readiness()[0]\n            return {"engine": "ready" if ready else "unavailable", "loaded_in_seconds": None}\n        return runner.state()\n\n    def fits(self, runner):\n        """True when `runner` is already loaded, or declares its size and that much GPU memory is free."""\n        if _resident(runner) and runner.state()["engine"] in ("ready", "loading"):\n            return True\n        need = getattr(runner, "vram_gib", None)\n        if not need:\n            return False\n        free = self.free_memory()\n        return free is not None and free >= need + 2\n\n    def acquire(self, kind, cancelled, progress):\n        """Make the engine for `kind` resident (swapping if needed) and return its runner."""\n        runner = self.runner(kind)\n        if runner is None:\n            raise RuntimeError(f"This engine has no {kind} model.")\n        with self.lock:\n            # Mark the switch first: readiness() of the outgoing kind must not restart it meanwhile.\n            self.active = kind\n            if self.swap and not self.fits(runner):\n                for other in self.runners.values():\n                    if other is not runner and _resident(other) and other.state()["engine"] != "stopped":\n                        progress(None, f"Unloading the {getattr(other, \'label\', \'other\')} model to make room…")\n                        other.stop()\n            if not _resident(runner):\n                return runner  # command-line and demo runners load per job\n            if runner.state()["engine"] in ("stopped", "failed"):\n                runner.start()\n        started = time.monotonic()\n        while runner.state()["engine"] == "loading":\n            if cancelled.wait(.5):\n                raise Cancelled()\n            progress(None, f"Loading the {kind} model ({int(time.monotonic() - started)} s)… this happens after switching models.")\n        ready, message = runner.readiness()\n        if not ready:\n            raise RuntimeError(message)\n        return runner\n', 'backend/matting.py': '"""Background removal. Only this module knows the matting model and how it is run.\n\nA matter turns one image into a same-sized grayscale mask: white is the subject (kept), black is\nbackground (hidden). The studio stores that as an ordinary layer mask, so no pixels are erased.\n"""\nimport os\nimport time\nfrom pathlib import Path\nfrom threading import RLock, Thread\nfrom typing import Protocol\nfrom PIL import Image, ImageChops, ImageFilter, ImageStat\n\nMATTING_MODEL = "onnx-community/BiRefNet-ONNX/onnx/model.onnx"\nMEAN = (0.485, 0.456, 0.406)\nSTD = (0.229, 0.224, 0.225)\n\n\nclass Matter(Protocol):\n    """Required: mode, model, readiness(), matte(). Optional: start(), stop(), state()."""\n    mode: str\n    model: str\n    def readiness(self) -> tuple[bool, str]: ...\n    def matte(self, image: Image.Image) -> Image.Image: ...\n\n\nclass BiRefNetMatter:\n    """BiRefNet (MIT) through ONNX Runtime; CUDA when available, CPU otherwise.\n\n    The session loads once in the background at service start, like the image engine, and is\n    reused for every request. Requests run on the service\'s single GPU worker, so matting never\n    competes with an image generation for GPU memory.\n    """\n    mode = "birefnet"\n\n    def __init__(self):\n        self.path = os.environ.get("ZERO_MATTING_MODEL", "")\n        self.model = os.environ.get("ZERO_MATTING_MODEL_ID", MATTING_MODEL)\n        self.size = int(os.environ.get("ZERO_MATTING_SIZE", "1024"))\n        self.device_preference = os.environ.get("ZERO_MATTING_DEVICE", "auto")  # auto | cuda | cpu\n        self.lock = RLock()\n        self.session = None\n        self.status = "stopped"  # stopped | loading | ready | failed\n        self.detail = ""\n        self.device = None\n        self.loaded_in = None\n\n    def start(self):\n        with self.lock:\n            if self.status in ("loading", "ready") or not Path(self.path).is_file():\n                return\n            self.status, self.detail = "loading", ""\n        Thread(target=self._load, daemon=True, name="studio-zero-matting").start()\n\n    def _providers(self, cpu_only=False):\n        import onnxruntime as ort\n        available = ort.get_available_providers()\n        if not cpu_only and self.device_preference != "cpu" and "CUDAExecutionProvider" in available:\n            return ["CUDAExecutionProvider", "CPUExecutionProvider"]\n        if self.device_preference == "cuda" and not cpu_only:\n            raise RuntimeError("CUDA was requested for background removal but ONNX Runtime has no CUDA provider.")\n        return ["CPUExecutionProvider"]\n\n    def _session(self, cpu_only=False):\n        import onnxruntime as ort\n        if not cpu_only and hasattr(ort, "preload_dlls"):\n            try:\n                ort.preload_dlls()  # find CUDA/cuDNN from pip-installed NVIDIA packages (e.g. on Colab)\n            except Exception:\n                pass\n        options = ort.SessionOptions()\n        options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL\n        session = ort.InferenceSession(self.path, sess_options=options, providers=self._providers(cpu_only))\n        device = "cuda" if "CUDAExecutionProvider" in session.get_providers() else "cpu"\n        return session, device\n\n    def _load(self):\n        started = time.monotonic()\n        try:\n            session, device = self._session()\n            with self.lock:\n                self.session, self.device = session, device\n                self.status, self.loaded_in = "ready", round(time.monotonic() - started, 1)\n        except Exception as exc:  # reported through /health; the image engine keeps working\n            with self.lock:\n                self.status, self.detail = "failed", f"Background removal could not load: {exc}"\n\n    def stop(self):\n        with self.lock:\n            self.session, self.status = None, "stopped"\n\n    def state(self):\n        with self.lock:\n            return {"matting": self.status, "matting_device": self.device}\n\n    def readiness(self):\n        if not self.path or not Path(self.path).is_file():\n            return False, "Background removal needs its model. Enable it in the Colab notebook and restart the API."\n        with self.lock:\n            status = self.status\n        if status == "stopped":\n            self.start()\n            status = "loading"\n        if status == "ready":\n            return True, f"Background removal ready ({self.device})"\n        if status == "loading":\n            return False, "Loading the background removal model. Try again in a moment."\n        return False, self.detail or "Background removal is unavailable."\n\n    def _tensor(self, image: Image.Image, dtype):\n        import numpy as np\n        rgb = image.convert("RGB").resize((self.size, self.size), Image.Resampling.BILINEAR)\n        array = (np.asarray(rgb, dtype=np.float32) / 255.0 - np.array(MEAN, dtype=np.float32)) / np.array(STD, dtype=np.float32)\n        return array.transpose(2, 0, 1)[None].astype(dtype)\n\n    def matte(self, image: Image.Image) -> Image.Image:\n        import numpy as np\n        ready, message = self.readiness()\n        if not ready:\n            raise RuntimeError(message)\n        with self.lock:\n            session = self.session\n        source = session.get_inputs()[0]\n        dtype = np.float16 if "float16" in source.type else np.float32\n        feed = {source.name: self._tensor(image, dtype)}\n        try:\n            logits = session.run([session.get_outputs()[0].name], feed)[0]\n        except Exception:\n            if self.device != "cuda":\n                raise\n            # Usually GPU memory pressure next to the image engine: fall back to the CPU once.\n            session, device = self._session(cpu_only=True)\n            with self.lock:\n                self.session, self.device = session, device\n            logits = session.run([session.get_outputs()[0].name], feed)[0]\n        logits = np.asarray(logits, dtype=np.float32).reshape(logits.shape[-2:])\n        alpha = 1.0 / (1.0 + np.exp(-np.clip(logits, -40, 40)))\n        mask = Image.fromarray(np.round(alpha * 255).astype(np.uint8), "L")\n        return mask.resize(image.size, Image.Resampling.BILINEAR)\n\n\nclass DemoMatter:\n    """Offline stand-in for tests and the demo service: separates colours that differ from the\n    image border. Clearly labelled; it is not AI background removal."""\n    mode = "demo"\n    model = "studio-zero/border-colour-demo"\n\n    def readiness(self):\n        return True, "Demo background removal — border colour key, not AI"\n\n    def matte(self, image: Image.Image) -> Image.Image:\n        rgb = image.convert("RGB")\n        w, h = rgb.size\n        strips = [rgb.crop(box) for box in ((0, 0, w, 1), (0, h - 1, w, h), (0, 0, 1, h), (w - 1, 0, w, h))]\n        border = Image.new("RGB", (w * 2 + h * 2, 1))\n        x = 0\n        for strip in strips:\n            flat = strip.resize((strip.width * strip.height, 1))\n            border.paste(flat, (x, 0))\n            x += flat.width\n        background = tuple(int(v) for v in ImageStat.Stat(border).median)\n        difference = ImageChops.difference(rgb, Image.new("RGB", rgb.size, background))\n        channels = difference.split()\n        distance = ImageChops.lighter(ImageChops.lighter(channels[0], channels[1]), channels[2])\n        mask = distance.point(lambda v: 0 if v < 24 else 255 if v > 56 else round((v - 24) * 255 / 32))\n        if image.mode in ("RGBA", "LA") or "transparency" in image.info:\n            mask = ImageChops.multiply(mask, image.convert("RGBA").getchannel("A"))\n        return mask.filter(ImageFilter.GaussianBlur(1))\n\n\ndef default_matter(runner_mode: str):\n    """Demo services get the demo matter; real engines need ZERO_MATTING_MODEL."""\n    if runner_mode == "demo":\n        return DemoMatter()\n    if os.environ.get("ZERO_MATTING_MODEL"):\n        return BiRefNetMatter()\n    return None\n', 'backend/model_cache.py': '"""Persistent model store: download each weight file once, reuse it in every later session.\n\nThe store is any directory that outlives the runtime (Google Drive on Colab, a disk on a GPU host):\n\n    <store>/models/<owner>/<repo>/<filename>\n    <store>/models/manifest.json   {"version": 1, "files": {"<repo>:<filename>": {"revision", "size"}}}\n\nA file counts as cached only when the manifest lists it and the stored copy has the recorded size,\nso a copy cut short by a runtime reset is detected and fetched again. Cached repositories stay pinned\nto their recorded revision: with everything cached, preparing makes no network request at all.\n`update=True` checks the repository\'s `main` branch and replaces only files that changed.\n`redownload=True` ignores the cache and fetches every file again (at the pinned revision unless\n`update=True`), replacing the stored copies.\n\nNew downloads land on fast local disk first; copying them into the store happens in a background\nthread so the engine can start immediately. Nothing here depends on Hugging Face directly: the\ncaller supplies `resolve` (repository metadata) and `download` (fetch one file) functions.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport shutil\nimport threading\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Callable, Iterable\n\nMANIFEST_VERSION = 1\n\n\n@dataclass(frozen=True)\nclass ModelFile:\n    env: str        # environment variable that receives the local path\n    repo: str       # "owner/name"\n    filename: str   # path inside the repository, e.g. "vae/model.safetensors"\n\n\n@dataclass(frozen=True)\nclass RepoSnapshot:\n    revision: str\n    sizes: dict[str, int | None] = field(default_factory=dict)  # filename -> bytes (None if unknown)\n\n\nResolver = Callable[[str, str], RepoSnapshot]        # (repo, revision or "main")\nDownloader = Callable[[str, str, str, Path], Path]   # (repo, filename, revision, destination dir) -> file\nLegacy = Callable[[ModelFile], Path]                 # where an older notebook layout kept the file\n\n\ndef _size(path: Path) -> int | None:\n    try:\n        return path.stat().st_size if path.is_file() else None\n    except OSError:\n        return None\n\n\ndef _copy(source: Path, target: Path) -> None:\n    """Copy through a temporary name so a partial copy never sits under the final name."""\n    target.parent.mkdir(parents=True, exist_ok=True)\n    partial = target.with_name(target.name + \'.partial\')\n    with source.open(\'rb\') as reader, partial.open(\'wb\') as writer:\n        shutil.copyfileobj(reader, writer, 16 * 1024 * 1024)\n    os.replace(partial, target)\n\n\nclass ModelStore:\n    def __init__(self, local: Path, store: Path | None = None, *, update: bool = False, redownload: bool = False,\n                 copy_to_local: bool = True, reserve_bytes: int = 256 * 1024 * 1024,\n                 log: Callable[[str], None] = print):\n        self.local = Path(local)\n        self.store = Path(store) if store else None\n        self.update = update\n        self.redownload = redownload\n        self.copy_to_local = copy_to_local\n        self.reserve_bytes = reserve_bytes\n        self.log = log\n        self.revisions: dict[str, str] = {}\n        self.paths: dict[tuple[str, str], str] = {}  # (repo, filename) → where this session reads it\n        self.save_errors: list[str] = []\n        self.reused: list[str] = []\n        self.downloaded: list[str] = []\n        self._pending: list[tuple[ModelFile, Path, str, int]] = []\n        self._lock = threading.Lock()\n        self._saver: threading.Thread | None = None\n\n    # ── manifest ──────────────────────────────────────────────────────────\n    @property\n    def manifest_path(self) -> Path | None:\n        return self.store / \'models\' / \'manifest.json\' if self.store else None\n\n    def stored_path(self, item: ModelFile) -> Path:\n        assert self.store is not None\n        return self.store / \'models\' / item.repo / item.filename\n\n    def read_manifest(self) -> dict:\n        path = self.manifest_path\n        if not path or not path.is_file():\n            return {\'version\': MANIFEST_VERSION, \'files\': {}}\n        try:\n            data = json.loads(path.read_text(encoding=\'utf-8\'))\n            if data.get(\'version\') != MANIFEST_VERSION or not isinstance(data.get(\'files\'), dict):\n                raise ValueError(\'unsupported manifest\')\n            return data\n        except (OSError, ValueError) as exc:\n            self.log(f\'Model cache manifest unreadable ({exc}); cached files will be verified again.\')\n            return {\'version\': MANIFEST_VERSION, \'files\': {}}\n\n    def _record(self, item: ModelFile, revision: str, size: int) -> None:\n        with self._lock:\n            manifest = self.read_manifest()\n            manifest[\'files\'][f\'{item.repo}:{item.filename}\'] = {\'revision\': revision, \'size\': size}\n            path = self.manifest_path\n            path.parent.mkdir(parents=True, exist_ok=True)\n            temporary = path.with_name(path.name + \'.partial\')\n            temporary.write_text(json.dumps(manifest, indent=2, sort_keys=True), encoding=\'utf-8\')\n            os.replace(temporary, path)\n\n    def _cached_revision(self, item: ModelFile, manifest: dict) -> str | None:\n        """The recorded revision if the stored copy is complete, else None."""\n        if not self.store:\n            return None\n        entry = manifest[\'files\'].get(f\'{item.repo}:{item.filename}\')\n        if not isinstance(entry, dict) or not isinstance(entry.get(\'revision\'), str):\n            return None\n        return entry[\'revision\'] if _size(self.stored_path(item)) == entry.get(\'size\') else None\n\n    # ── preparing ─────────────────────────────────────────────────────────\n    def prepare(self, files: Iterable[ModelFile], resolve: Resolver, download: Downloader,\n                legacy: Legacy | None = None) -> dict[str, str]:\n        """Return {env: local path} for every file, reusing the store and downloading only what is missing."""\n        files = list(files)\n        manifest = self.read_manifest()\n        paths: dict[str, str] = {}\n        by_repo: dict[str, list[ModelFile]] = {}\n        for item in files:\n            by_repo.setdefault(item.repo, []).append(item)\n\n        for repo, items in by_repo.items():\n            cached = {item: self._cached_revision(item, manifest) for item in items}\n            pinned = sorted({rev for rev in cached.values() if rev})\n            snapshot = None\n            if self.update or not pinned:\n                snapshot = resolve(repo, \'main\')\n            elif self.redownload or any(cached[item] != pinned[0] for item in items):\n                snapshot = resolve(repo, pinned[0])   # add a file to a pinned repository at the same revision\n            revision = snapshot.revision if snapshot else pinned[0]\n            self.revisions[repo] = revision\n\n            for item in items:\n                if cached[item] == revision and not self.redownload:\n                    paths[item.env] = self.paths[(item.repo, item.filename)] = str(self._use_stored(item))\n                    self.reused.append(item.filename)\n                    continue\n                if item.filename not in snapshot.sizes:\n                    raise RuntimeError(f\'{item.filename} is not in {repo} at {revision[:12]}. \'\n                                       \'The repository layout changed; update the notebook settings.\')\n                expected = snapshot.sizes[item.filename]\n                adopted = self._adopt_legacy(item, legacy, revision, expected) if (legacy and not self.redownload) else None\n                if adopted:\n                    paths[item.env] = self.paths[(item.repo, item.filename)] = str(adopted)\n                    self.reused.append(item.filename)\n                    continue\n                self.log(f\'Downloading {repo}/{item.filename}…\')\n                path = Path(download(repo, item.filename, revision, self.local / repo))\n                actual = _size(path)\n                if actual is None or (expected is not None and actual != expected):\n                    raise RuntimeError(f\'Download of {item.filename} is incomplete ({actual} of {expected} bytes).\')\n                paths[item.env] = self.paths[(item.repo, item.filename)] = str(path)\n                self.downloaded.append(item.filename)\n                if self.store:\n                    self._pending.append((item, path, revision, actual))\n\n        paths.pop("", None)  # files without their own variable (whole-folder snapshots) are in self.paths\n        if self._pending:\n            self._saver = threading.Thread(target=self._save_pending, name=\'model-store-save\', daemon=True)\n            self._saver.start()\n        return paths\n\n    def _use_stored(self, item: ModelFile) -> Path:\n        stored = self.stored_path(item)\n        if not self.copy_to_local:\n            return stored\n        local = self.local / item.repo / item.filename\n        if _size(local) != _size(stored):\n            self.log(f\'Restoring {item.filename} from the cache…\')\n            _copy(stored, local)\n        return local\n\n    def _adopt_legacy(self, item: ModelFile, legacy: Legacy, revision: str, expected: int | None) -> Path | None:\n        """Move a complete file from an older cache layout into the store instead of downloading it again."""\n        if not self.store:\n            return None\n        old = legacy(item)\n        size = _size(old)\n        if size is None or expected is None or size != expected:\n            return None\n        target = self.stored_path(item)\n        target.parent.mkdir(parents=True, exist_ok=True)\n        os.replace(old, target)\n        self._record(item, revision, size)\n        self.log(f\'Reused {item.filename} from the previous Drive cache layout.\')\n        return self._use_stored(item)\n\n    # ── saving into the store ─────────────────────────────────────────────\n    def _save_pending(self) -> None:\n        for item, path, revision, size in self._pending:\n            try:\n                free = shutil.disk_usage(self.store).free\n            except OSError:\n                free = None\n            if free is not None and free < size + self.reserve_bytes:\n                message = (f\'Not enough space in the cache for {item.filename} \'\n                           f\'({size / 1e9:.2f} GB needed, {free / 1e9:.2f} GB free); it will download again next session.\')\n                self.save_errors.append(message); self.log(message)\n                continue\n            try:\n                _copy(path, self.stored_path(item))\n                if _size(self.stored_path(item)) != size:\n                    raise OSError(\'stored copy has the wrong size\')\n                self._record(item, revision, size)\n                self.log(f\'Saved {item.filename} to the cache.\')\n            except OSError as exc:\n                message = f\'Could not save {item.filename} to the cache: {exc}\'\n                self.save_errors.append(message); self.log(message)\n\n    @property\n    def saving(self) -> bool:\n        return bool(self._saver and self._saver.is_alive())\n\n    def wait(self, timeout: float | None = None) -> bool:\n        """Block until background saves finish. Returns False if they are still running."""\n        if self._saver:\n            self._saver.join(timeout)\n        return not self.saving\n\n    def summary(self) -> str:\n        parts = [f\'{len(self.reused)} reused from cache\', f\'{len(self.downloaded)} downloaded\']\n        if self.saving:\n            parts.append(\'saving new files to the cache in the background\')\n        if self.save_errors:\n            parts.append(f\'{len(self.save_errors)} not cached (see messages above)\')\n        return \', \'.join(parts)\n\n\ndef link_view(paths: dict, repo: str, destination: Path) -> Path:\n    """One folder with a repository\'s layout, linking each file wherever this session reads it\n    (Drive or local disk). Libraries that load a whole folder (diffusers) then see a normal snapshot."""\n    destination = Path(destination)\n    for (owner, filename), path in paths.items():\n        if owner != repo:\n            continue\n        target = destination / filename\n        target.parent.mkdir(parents=True, exist_ok=True)\n        if target.is_symlink() or target.exists():\n            target.unlink()\n        target.symlink_to(Path(path).resolve())\n    return destination\n', 'backend/runner.py': '"""Only this module knows the model runner\'s command-line and local engine interfaces."""\nimport base64\nimport io\nimport json\nimport math\nimport os\nimport random\nimport shutil\nimport subprocess\nimport tempfile\nimport time\nimport urllib.error\nimport urllib.request\nfrom pathlib import Path\nfrom threading import Event, RLock, Thread\nfrom typing import Callable, Protocol\nfrom PIL import Image, ImageChops, ImageDraw, ImageFilter, PngImagePlugin\nfrom .schema import EditRequest, GenerateRequest, VideoRequest, decode_image_data\n\nMODEL = "abenzerps/Qwen-Image-2.1-GGUF/qwen-image-2.1-UC-Q4_K_M"\nVIDEO_MODEL = "vantagewithai/LTX-2.5-GGUF/ltx-2.5-22b-distilled-transformer-Q4_K_M"\nSD_REVISION = "2bb72947cb129962f350452148658a32f4d3c057"\n\n\nclass Cancelled(Exception):\n    pass\n\n\nclass Runner(Protocol):\n    """Required: mode, model, readiness(), generate(). Optional: edit(), start(), stop(), state()."""\n    mode: str\n    model: str\n    def readiness(self) -> tuple[bool, str]: ...\n    def generate(self, request: GenerateRequest, output: Path, cancelled: Event, progress: Callable) -> None: ...\n\n\ndef transparent_prompt(prompt: str) -> str:\n    """The documented Qwen-Image 2.1 phrasing for RGBA output with a transparent background."""\n    return ("This is an RGBA image with transparency. " + prompt.strip().rstrip(".") +\n            ". The image has alpha channel and the background is transparent.")\n\n\n_gpu_memory: list = []\n\n\ndef gpu_memory_gib():\n    """Total memory of the first GPU in GiB, or None if it cannot be read (cached)."""\n    if not _gpu_memory:\n        try:\n            text = subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],\n                                  capture_output=True, text=True, timeout=20, check=True).stdout\n            _gpu_memory.append(float(text.split()[0]) / 1024)\n        except (OSError, subprocess.SubprocessError, ValueError, IndexError):\n            _gpu_memory.append(None)\n    return _gpu_memory[0]\n\n\ndef offload_wanted(weight_paths, env="ZERO_OFFLOAD"):\n    """on/off force CPU offload; auto (default) offloads only when the weights plus headroom exceed GPU memory."""\n    mode = os.environ.get(env, "auto").strip().lower()\n    if mode in ("1", "on", "true", "yes"):\n        return True\n    if mode in ("0", "off", "false", "no"):\n        return False\n    vram = gpu_memory_gib()\n    if vram is None:\n        return True\n    total = 0\n    for path in weight_paths:\n        try:\n            total += Path(path).stat().st_size\n        except OSError:\n            return True\n    headroom = float(os.environ.get("ZERO_VRAM_HEADROOM_GIB", "6"))\n    return total / 2**30 + headroom > vram\n\n\ndef find_ffmpeg():\n    """ffmpeg converts the engine\'s AVI (MJPEG + PCM audio) into browser-playable MP4."""\n    configured = os.environ.get("ZERO_FFMPEG")\n    if configured and Path(configured).is_file():\n        return configured\n    found = shutil.which("ffmpeg")\n    if found:\n        return found\n    try:\n        import imageio_ffmpeg  # optional fallback\n        return imageio_ffmpeg.get_ffmpeg_exe()\n    except Exception:\n        return None\n\n\ndef video_format():\n    """mp4 (H.264 + AAC, plays everywhere, default) or webm (VP9 + Opus, royalty-free)."""\n    return "webm" if os.environ.get("ZERO_VIDEO_FORMAT", "mp4").lower() == "webm" else "mp4"\n\n\ndef encoder_args(fmt):\n    if fmt == "webm":\n        return ["-c:v", "libvpx-vp9", "-b:v", "0", "-crf", "30", "-row-mt", "1", "-deadline", "realtime", "-cpu-used", "8",\n                "-pix_fmt", "yuv420p", "-c:a", "libopus", "-b:a", "128k"]\n    return ["-c:v", "libx264", "-preset", "veryfast", "-crf", "18", "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "160k",\n            "-movflags", "+faststart"]\n\n\ndef to_browser_video(source: Path, output: Path, cancelled: Event, fmt=None):\n    """Convert the engine\'s AVI (MJPEG + PCM audio) into a browser-playable MP4 or WebM."""\n    fmt = fmt or video_format()\n    ffmpeg = find_ffmpeg()\n    if not ffmpeg:\n        raise RuntimeError("Video needs ffmpeg on the engine to convert the result. Install ffmpeg and restart the API.")\n    temporary = output.with_name(output.stem + ".tmp." + fmt)\n    process = subprocess.Popen([ffmpeg, "-y", "-loglevel", "error", "-i", str(source), *encoder_args(fmt), str(temporary)],\n                               stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, shell=False)\n    while process.poll() is None:\n        if cancelled.wait(.2):\n            process.kill(); process.wait(); temporary.unlink(missing_ok=True)\n            raise Cancelled()\n    if process.returncode or not temporary.is_file():\n        temporary.unlink(missing_ok=True)\n        raise RuntimeError("Could not convert the video for the browser: " + (process.stderr.read().decode(errors="replace")[-300:] if process.stderr else ""))\n    temporary.replace(output)\n\n\ndef video_info(path: Path):\n    """(width, height, decoded frame count, fps) of a video, parsed from ffmpeg\'s own output (no ffprobe needed)."""\n    ffmpeg = find_ffmpeg()\n    result = subprocess.run([ffmpeg, "-hide_banner", "-i", str(path), "-map", "0:v:0", "-f", "null", "-"],\n                            capture_output=True, text=True, timeout=120)\n    text = result.stderr\n    import re\n    size = re.search(r"Video: [^\\n]*?(\\d{2,5})x(\\d{2,5})", text)\n    fps = re.search(r"(\\d+(?:\\.\\d+)?) fps", text)\n    frames = re.findall(r"frame=\\s*(\\d+)", text)\n    if not size:\n        raise RuntimeError("The engine produced an unreadable video.")\n    return int(size.group(1)), int(size.group(2)), int(frames[-1]) if frames else 0, float(fps.group(1)) if fps else 0.0\n\n\n# Command-line runner ------------------------------------------------------------------------------\n\nclass _QwenWeights:\n    """Weight paths shared by every Qwen runner; configured only through the environment."""\n    mode = "qwen"\n    model = MODEL\n    binary_env = "ZERO_SD_CLI"\n    binary_label = "runner"\n\n    def __init__(self):\n        self.binary = os.environ.get(self.binary_env, "")\n        self.diffusion = os.environ.get("ZERO_DIFFUSION", "")\n        self.encoder = os.environ.get("ZERO_TEXT_ENCODER", "")\n        self.vae = os.environ.get("ZERO_VAE", "")\n        self.model = os.environ.get("ZERO_MODEL_ID", MODEL)\n\n    def weights(self):\n        return [("diffusion weights", self.diffusion, "--diffusion-model"), ("text encoder", self.encoder, "--llm"), ("VAE", self.vae, "--vae")]\n\n    def missing(self):\n        items = [(self.binary_label, self.binary), *[(label, path) for label, path, _ in self.weights()]]\n        return [name for name, value in items if not value or not Path(value).is_file()]\n\n    def weight_args(self):\n        return [part for _, path, flag in self.weights() for part in (flag, path)]\n\n\nclass QwenRunner(_QwenWeights):\n    """One sd-cli process per image. Simple and robust, but reloads ~10 GB of weights every time."""\n\n    def readiness(self):\n        missing = self.missing()\n        return (False, "Configure missing " + ", ".join(missing)) if missing else (True, "Qwen engine ready")\n\n    def command(self, r: GenerateRequest, output: Path):\n        # Explicit argv, never a shell. Prompts cannot become options or commands.\n        prompt = transparent_prompt(r.prompt) if r.transparent else r.prompt\n        return [self.binary, *self.weight_args(), "-p", prompt, "-n", r.negative_prompt,\n                "-W", str(r.width), "-H", str(r.height), "--steps", str(r.steps),\n                "--cfg-scale", str(r.guidance), "--seed", str(r.seed),\n                "--sampling-method", "euler", "--offload-to-cpu", "-o", str(output)]\n\n    def generate(self, request, output, cancelled, progress):\n        ready, message = self.readiness()\n        if not ready:\n            raise RuntimeError(message)\n        progress(None, "Creating your image…")\n        log_path = output.with_suffix(".log")\n        # Write logs to a private file rather than a pipe that can deadlock on verbose output.\n        with log_path.open("wb") as log:\n            process = subprocess.Popen(self.command(request, output), stdout=log, stderr=subprocess.STDOUT, shell=False)\n            try:\n                deadline = time.monotonic() + int(os.environ.get("ZERO_JOB_TIMEOUT", "3600"))\n                while process.poll() is None:\n                    if cancelled.wait(.2):\n                        raise Cancelled()\n                    if time.monotonic() > deadline:\n                        raise RuntimeError("Generation timed out. Try a smaller image or more GPU memory.")\n                if cancelled.is_set():\n                    raise Cancelled()\n                if process.returncode:\n                    raise RuntimeError("The image engine failed. Check the private service log for memory or model compatibility errors.")\n            finally:\n                if process.poll() is None:\n                    process.terminate()\n                    try:\n                        process.wait(timeout=5)\n                    except subprocess.TimeoutExpired:\n                        process.kill()\n                        process.wait()\n        if not output.is_file():\n            # Some sd-cli versions suffix the single result with _1.\n            alternative = output.with_name(output.stem + "_1.png")\n            if alternative.is_file():\n                alternative.replace(output)\n            else:\n                raise RuntimeError("Engine finished without producing an image.")\n\n\n# Resident engines (sd-server child processes) -----------------------------------------------------\n\nclass ResidentEngine:\n    """Loads weights once into a private sd-server child process and reuses them for every job.\n\n    The child listens only on 127.0.0.1 and is never exposed: the authenticated Studio Zero API\n    remains the single public surface. Loading starts in the background, so the API answers\n    immediately and reports "loading" until the model is resident. Subclasses supply weights.\n    """\n    binary_env = "ZERO_SD_SERVER"\n    binary_label = "engine server"\n    port_env, default_port = "ZERO_ENGINE_PORT", 18431\n    log_name = "engine.log"\n    label = "image"\n\n    def _init_engine(self):\n        self.binary = os.environ.get(self.binary_env, "")\n        self.port = int(os.environ.get(self.port_env, str(self.default_port)))\n        self.base = f"http://127.0.0.1:{self.port}"\n        self.flash_attention = os.environ.get("ZERO_FLASH_ATTENTION", "1") != "0"\n        self.sage_attention = os.environ.get("ZERO_SAGE_ATTENTION", "0") == "1"\n        self.load_timeout = int(os.environ.get("ZERO_ENGINE_LOAD_TIMEOUT", "1800"))\n        log_dir = Path(os.environ.get("ZERO_OUTPUT_DIR", "outputs"))\n        default_log = os.environ.get("ZERO_ENGINE_LOG") if self.log_name == "engine.log" else None\n        self.log_path = Path(default_log or log_dir / self.log_name)\n        self.lock = RLock()\n        self.process = None\n        self.status = "stopped"  # stopped | loading | ready | failed\n        self.detail = ""\n        self.loaded_in = None\n        self.last_start = 0.0\n\n    # Configuration ---------------------------------------------------------\n    def weights(self):  # [(label, path, flag)]\n        raise NotImplementedError\n\n    def extra_args(self):\n        return []\n\n    def missing(self):\n        items = [(self.binary_label, self.binary), *[(label, path) for label, path, _ in self.weights()]]\n        return [name for name, value in items if not value or not Path(value).is_file()]\n\n    def weight_args(self):\n        return [part for _, path, flag in self.weights() for part in (flag, path)]\n\n    @property\n    def offload(self):\n        return offload_wanted([path for _, path, _ in self.weights()])\n\n    def command(self):\n        argv = [self.binary, *self.weight_args(), *self.extra_args()]\n        if self.offload:\n            argv.append("--offload-to-cpu")\n        argv += ["--listen-ip", "127.0.0.1", "--listen-port", str(self.port)]\n        if self.flash_attention:\n            argv.append("--diffusion-fa")\n        if self.sage_attention:\n            argv.append("--sage-attn")\n        return argv\n\n    # Lifecycle ---------------------------------------------------------------\n    def start(self):\n        with self.lock:\n            if self.process is not None and self.process.poll() is None:\n                return\n            if self.missing():\n                return\n            self.log_path.parent.mkdir(parents=True, exist_ok=True)\n            log = self.log_path.open("ab")\n            self.process = subprocess.Popen(self.command(), stdout=log, stderr=subprocess.STDOUT, shell=False)\n            log.close()\n            self.status, self.detail, self.loaded_in = "loading", "", None\n            self.last_start = time.monotonic()\n            Thread(target=self._await_ready, args=(self.process,), daemon=True, name=f"studio-zero-{self.label}-engine").start()\n\n    def _await_ready(self, process):\n        started = time.monotonic()\n        while time.monotonic() - started < self.load_timeout:\n            if process.poll() is not None:\n                with self.lock:\n                    if self.process is process:\n                        self.status, self.detail = "failed", f"The {self.label} engine stopped while loading (exit {process.returncode}). Check {self.log_path.name}."\n                return\n            try:\n                self._request("GET", "/sdcpp/v1/capabilities", timeout=2)\n                with self.lock:\n                    if self.process is process:\n                        self.status, self.loaded_in = "ready", round(time.monotonic() - started, 1)\n                return\n            except (OSError, ValueError):\n                time.sleep(1)\n        with self.lock:\n            if self.process is not process:\n                return\n            self.status, self.detail = "failed", f"The {self.label} engine did not finish loading in time. Check {self.log_path.name}."\n        self._terminate(process)\n\n    def stop(self):\n        with self.lock:\n            process, self.process = self.process, None\n            self.status = "stopped"\n        self._terminate(process)\n\n    @staticmethod\n    def _terminate(process):\n        if process is not None and process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=15)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n\n    def state(self):\n        with self.lock:\n            if self.status == "ready" and (self.process is None or self.process.poll() is not None):\n                code = self.process.returncode if self.process is not None else "?"\n                self.status, self.detail = "failed", f"The {self.label} engine stopped unexpectedly (exit {code}). Check {self.log_path.name}."\n            return {"engine": self.status, "loaded_in_seconds": self.loaded_in}\n\n    def readiness(self):\n        missing = self.missing()\n        if missing:\n            return False, "Configure missing " + ", ".join(missing)\n        status = self.state()["engine"]\n        # Start lazily, and restart a stopped/crashed engine at most once every 60 seconds.\n        if status == "stopped" or (status == "failed" and time.monotonic() - self.last_start > 60):\n            self.start()\n            status = self.state()["engine"]\n        if status == "ready":\n            return True, self.ready_message\n        if status == "loading":\n            return False, f"Loading the {self.label} model into memory. This happens once per engine start."\n        return False, self.detail or f"The {self.label} engine is not running."\n\n    ready_message = "Engine ready"\n\n    # Jobs ----------------------------------------------------------------------\n    def _request(self, method, path, body=None, timeout=30):\n        data = None if body is None else json.dumps(body).encode()\n        req = urllib.request.Request(self.base + path, data=data, method=method, headers={"Content-Type": "application/json"} if data else {})\n        with urllib.request.urlopen(req, timeout=timeout) as response:\n            return json.loads(response.read() or b"{}")\n\n    def _run(self, endpoint, payload, cancelled, progress, working, timeout_message):\n        ready, message = self.readiness()\n        if not ready:\n            raise RuntimeError(message)\n        try:\n            job = self._request("POST", endpoint, payload, timeout=120)\n        except urllib.error.HTTPError as exc:\n            detail = ""\n            try:\n                detail = json.loads(exc.read() or b"{}").get("error", {}).get("message", "")\n            except (ValueError, AttributeError):\n                pass\n            raise RuntimeError(f"The engine rejected this request ({exc.code}{\': \' + detail if detail else \'\'}). Check {self.log_path.name}.") from None\n        except OSError:\n            raise RuntimeError(f"The engine is not reachable. Check {self.log_path.name}.") from None\n        job_id = job["id"]\n        deadline = time.monotonic() + int(os.environ.get("ZERO_JOB_TIMEOUT", "3600"))\n        while True:\n            if cancelled.wait(.5):\n                # Queued engine jobs are removed; one already on the GPU finishes and is discarded.\n                try:\n                    self._request("POST", f"/sdcpp/v1/jobs/{job_id}/cancel", {}, timeout=5)\n                except (OSError, ValueError):\n                    pass\n                raise Cancelled()\n            if time.monotonic() > deadline:\n                raise RuntimeError(timeout_message)\n            if self.state()["engine"] != "ready":\n                raise RuntimeError(self.detail or f"The engine stopped during generation. Check {self.log_path.name}.")\n            try:\n                job = self._request("GET", f"/sdcpp/v1/jobs/{job_id}", timeout=10)\n            except (OSError, ValueError):\n                continue\n            status = job.get("status")\n            if status == "completed":\n                return job.get("result") or {}\n            if status == "failed":\n                raise RuntimeError(f"The {self.label} engine failed: " + str((job.get("error") or {}).get("message", "unknown error")))\n            if status == "cancelled":\n                raise Cancelled()\n            position = job.get("queue_position") or 0\n            progress(None, working if status == "generating" or position == 0 else "Waiting for the engine…")\n\n\nclass ResidentQwenRunner(_QwenWeights, ResidentEngine):\n    """Qwen-Image 2.1 text-to-image, reference editing, inpainting and image-to-image."""\n    binary_env = "ZERO_SD_SERVER"\n    binary_label = "engine server"\n    ready_message = "Qwen engine ready — model loaded and kept in memory"\n\n    def __init__(self):\n        _QwenWeights.__init__(self)\n        self._init_engine()\n        self.vision = os.environ.get("ZERO_VISION_ENCODER", "")\n\n    @property\n    def supports_references(self):\n        return bool(self.vision and Path(self.vision).is_file())\n\n    supports_edit = True\n    supports_transparency = True\n\n    def extra_args(self):\n        return ["--llm_vision", self.vision] if self.supports_references else []\n\n    def vae_tiling(self, width, height):\n        mode = os.environ.get("ZERO_VAE_TILING", "auto").lower()\n        if mode in ("1", "on"):\n            return True\n        if mode in ("0", "off"):\n            return False\n        return width * height > int(os.environ.get("ZERO_VAE_TILING_PIXELS", str(1536 * 1536)))\n\n    def payload(self, r: GenerateRequest):\n        body = {"prompt": transparent_prompt(r.prompt) if r.transparent else r.prompt,\n                "negative_prompt": r.negative_prompt, "width": r.width, "height": r.height,\n                "ref_images": r.reference_images, "auto_resize_ref_image": False,\n                "seed": r.seed, "batch_count": 1, "embed_image_metadata": False, "output_format": "png",\n                "sample_params": {"sample_method": "euler", "sample_steps": r.steps, "guidance": {"txt_cfg": float(r.guidance)}}}\n        if self.vae_tiling(r.width, r.height):\n            body["vae_tiling_params"] = {"enabled": True}\n        return body\n\n    def edit_payload(self, r: EditRequest):\n        """Map a studio edit onto the engine\'s native fields (mask: white = regenerate, black = keep)."""\n        body = self.payload(r)\n        references = list(r.reference_images)\n        if r.operation == "edit":\n            body["ref_images"] = [r.image, *references]\n        else:\n            body["init_image"] = r.image\n            body["strength"] = r.effective_strength\n            if r.operation in ("inpaint", "outpaint"):\n                body["mask_image"] = r.mask\n            # With the vision encoder, the source also conditions the edit so new content matches it.\n            body["ref_images"] = ([r.image] if self.supports_references and r.operation != "image-to-image" else []) + references\n        return body\n\n    def _image(self, payload, output, cancelled, progress, working):\n        result = self._run("/sdcpp/v1/img_gen", payload, cancelled, progress, working,\n                           "Generation timed out. Try a smaller image or more GPU memory.")\n        images = result.get("images") or []\n        if not images:\n            raise RuntimeError("Engine finished without producing an image.")\n        output.write_bytes(base64.b64decode(images[0]["b64_json"]))\n\n    def generate(self, request, output, cancelled, progress):\n        self._image(self.payload(request), output, cancelled, progress, "Creating your image…")\n\n    def edit(self, request, output, cancelled, progress):\n        self._image(self.edit_payload(request), output, cancelled, progress, "Editing your image…")\n\n\nclass ResidentVideoRunner(ResidentEngine):\n    """LTX-2.5 text/image-to-video with audio through a second sd-server (vid_gen)."""\n    mode = "qwen"\n    label = "video"\n    port_env, default_port = "ZERO_VIDEO_ENGINE_PORT", 18432\n\n    @property\n    def format(self):\n        return video_format()\n    log_name = "video-engine.log"\n    ready_message = "Video engine ready — model loaded and kept in memory"\n\n    def __init__(self):\n        self.model = os.environ.get("ZERO_VIDEO_MODEL_ID", VIDEO_MODEL)\n        self.diffusion = os.environ.get("ZERO_VIDEO_DIFFUSION", "")\n        self.encoder = os.environ.get("ZERO_VIDEO_TEXT_ENCODER", "")\n        self.vae = os.environ.get("ZERO_VIDEO_VAE", "")\n        self.audio_vae = os.environ.get("ZERO_VIDEO_AUDIO_VAE", "")\n        self.connectors = os.environ.get("ZERO_VIDEO_CONNECTORS", "")\n        self._init_engine()\n\n    @property\n    def configured(self):\n        return bool(self.diffusion)\n\n    def weights(self):\n        items = [("video diffusion weights", self.diffusion, "--diffusion-model"), ("video text encoder", self.encoder, "--llm"),\n                 ("video VAE", self.vae, "--vae"), ("audio VAE", self.audio_vae, "--audio-vae")]\n        if self.connectors:\n            items.append(("embeddings connectors", self.connectors, "--embeddings-connectors"))\n        return items\n\n    def missing(self):\n        missing = super().missing()\n        return missing + ([] if find_ffmpeg() else ["ffmpeg"])\n\n    def payload(self, r: VideoRequest):\n        body = {"prompt": r.prompt, "negative_prompt": r.negative_prompt, "width": r.width, "height": r.height,\n                "seed": r.seed, "video_frames": r.frames, "fps": r.fps, "output_format": "avi", "output_compression": 95,\n                "embed_image_metadata": False,\n                "sample_params": {"sample_method": "euler", "sample_steps": r.steps, "guidance": {"txt_cfg": float(r.guidance)}},\n                "vae_tiling_params": {"enabled": True, "temporal_tiling": True}}\n        if r.image:\n            body["init_image"] = r.image\n        if r.end_image:\n            body["end_image"] = r.end_image\n        return body\n\n    def generate_video(self, request, output, cancelled, progress):\n        result = self._run("/sdcpp/v1/vid_gen", self.payload(request), cancelled, progress, "Creating your video…",\n                           "Video generation timed out. Try fewer frames, a smaller size or more GPU memory.")\n        data = result.get("b64_json")\n        if not data:\n            raise RuntimeError("The video engine finished without producing a video.")\n        progress(None, "Preparing the video for the browser…")\n        with tempfile.TemporaryDirectory() as folder:\n            source = Path(folder) / "engine.avi"\n            source.write_bytes(base64.b64decode(data))\n            to_browser_video(source, output, cancelled, self.format)\n\n\n# Demo engines --------------------------------------------------------------------------------------\n\nclass DemoRunner:\n    """Procedural graphics for offline integration testing. Never presented as AI."""\n    mode = "demo"\n    model = "studio-zero/procedural-demo"\n    supports_edit = True\n    supports_transparency = True\n\n    supports_references = False\n\n    def __init__(self, references=None):\n        # Off by default: a text-only demo must reject references, not ignore them. With references on\n        # (ZERO_DEMO_REFERENCES=1, for testing reference and pose flows) each reference is visibly stamped in.\n        if references is None and os.environ.get("ZERO_DEMO_REFERENCES") == "1":\n            references = True\n        if references is not None:\n            self.supports_references = references\n\n    def readiness(self):\n        return True, "Demo engine ready — procedural artwork, not AI generation"\n\n    @staticmethod\n    def _stamp_references(image, references):\n        """Paste each reference as a labelled thumbnail along the top edge, so it is visibly used."""\n        size = max(32, min(image.size) // 5)\n        for i, value in enumerate(references):\n            thumb = decode_image_data(value).convert("RGB")\n            ratio = size / max(thumb.size)\n            thumb = thumb.resize((max(1, round(thumb.width * ratio)), max(1, round(thumb.height * ratio))))\n            x = image.width - (i + 1) * (size + 8)\n            image.paste(thumb, (x, 8))\n            ImageDraw.Draw(image).text((x + 4, 10 + thumb.height), f"REF {i + 1}", fill="#ffffff")\n        return image\n\n    def _wait(self, cancelled, progress, message):\n        for i in range(8):\n            if cancelled.wait(.15):\n                raise Cancelled()\n            progress(i * 12, message)\n\n    @staticmethod\n    def _label(image, text):\n        draw = ImageDraw.Draw(image)\n        w, h = image.size\n        draw.text((int(w * .04), int(h * .92)), text, fill="#d2e6b4", font_size=max(12, w // 45))\n\n    @staticmethod\n    def _save(image, output):\n        info = PngImagePlugin.PngInfo(); info.add_text("Studio Zero", "Procedural demo. Not an AI-generated image.")\n        image.save(output, pnginfo=info)\n\n    def artwork(self, request):\n        rng = random.Random(request.seed)\n        w, h = request.width, request.height\n        if request.transparent:\n            im = Image.new("RGBA", (w, h), (0, 0, 0, 0))\n            draw = ImageDraw.Draw(im)\n            r = int(min(w, h) * .32)\n            cx, cy = w // 2, h // 2\n            draw.ellipse((cx - r, cy - r, cx + r, cy + r), fill=(88, 121, 96, 255))\n            draw.ellipse((cx - r // 3, cy - r // 2, cx + r // 6, cy - r // 6), fill=(228, 236, 182, 255))\n            for _ in range(5):\n                x, y = rng.randint(cx - r, cx + r), rng.randint(cy - r, cy + r)\n                draw.ellipse((x - 6, y - 6, x + 6, y + 6), fill=(53, 91, 73, 255))\n            return im\n        im = Image.new("RGB", (w, h), "#d4dbbf")\n        draw = ImageDraw.Draw(im)\n        for y in range(h):\n            t = y / h\n            draw.line((0, y, w, y), fill=(int(201 - 48 * t), int(217 - 42 * t), int(192 - 41 * t)))\n        sun_x, sun_y, radius = int(w * .7), int(h * .28), int(min(w, h) * .14)\n        draw.ellipse((sun_x - radius, sun_y - radius, sun_x + radius, sun_y + radius), fill="#e4ecb6")\n        for index, color in enumerate(["#879c7e", "#587960", "#355b49", "#20463d"]):\n            points = [(0, h), (0, h * (.52 + index * .1))]\n            points += [(x, h * (.52 + index * .1) + rng.randint(-int(h * .1), int(h * .1))) for x in range(0, w + 1, max(1, w // 6))]\n            points += [(w, h)]\n            draw.polygon(points, fill=color)\n        self._label(im, "STUDIO ZERO / DEMO ARTWORK")\n        return im\n\n    def generate(self, request, output, cancelled, progress):\n        self._wait(cancelled, progress, "Composing demo artwork…")\n        self._save(self._stamp_references(self.artwork(request), request.reference_images), output)\n\n    def edit(self, request, output, cancelled, progress):\n        """Visibly procedural: masked areas get a hatched demo fill; edits/variations are tinted."""\n        self._wait(cancelled, progress, "Applying demo edit…")\n        source = decode_image_data(request.image).convert("RGBA").resize((request.width, request.height))\n        tint = Image.new("RGBA", source.size, (120, 170, 220, 255) if request.operation in ("edit", "image-to-image", "variations") else (220, 140, 90, 255))\n        if request.operation in ("inpaint", "outpaint"):\n            mask = decode_image_data(request.mask).convert("L").resize(source.size)\n            fill = Image.new("RGBA", source.size, (220, 140, 90, 255))\n            draw = ImageDraw.Draw(fill)\n            for x in range(-source.height, source.width, 18):\n                draw.line((x, 0, x + source.height, source.height), fill=(240, 196, 120, 255), width=6)\n            result = Image.composite(fill, source, mask)\n        else:\n            result = Image.blend(source, tint, request.effective_strength * .45)\n            if request.transparent:\n                result.putalpha(source.getchannel("A"))\n        if not request.transparent:\n            result = result.convert("RGB")\n        self._label(result, "DEMO EDIT - NOT AI")\n        self._save(self._stamp_references(result, request.reference_images), output)\n\n\nclass DemoVideoRunner:\n    """Procedural moving shapes encoded to MP4 with ffmpeg. Never presented as AI."""\n    mode = "demo"\n    label = "video"\n    model = "studio-zero/procedural-demo-video"\n\n    @property\n    def format(self):\n        return video_format()\n\n    def readiness(self):\n        return (True, "Demo video engine ready — procedural animation, not AI") if find_ffmpeg() else (False, "Demo video needs ffmpeg.")\n\n    def missing(self):\n        return [] if find_ffmpeg() else ["ffmpeg"]\n\n    def generate_video(self, request, output, cancelled, progress):\n        ffmpeg = find_ffmpeg()\n        if not ffmpeg:\n            raise RuntimeError("Demo video needs ffmpeg.")\n        start = decode_image_data(request.image).convert("RGB").resize((request.width, request.height)) if request.image else None\n        w, h = request.width, request.height\n        temporary = output.with_name(output.stem + ".tmp." + self.format)\n        video_args = [a for a in encoder_args(self.format)]\n        # No audio track in the demo: drop the audio encoder options.\n        for flag in ("-c:a", "-b:a"):\n            i = video_args.index(flag)\n            del video_args[i:i + 2]\n        process = subprocess.Popen([ffmpeg, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{w}x{h}",\n                                    "-r", str(request.fps), "-i", "-", *video_args, str(temporary)], stdin=subprocess.PIPE, stderr=subprocess.DEVNULL)\n        try:\n            for i in range(request.frames):\n                if cancelled.is_set():\n                    raise Cancelled()\n                if i % 8 == 0:\n                    progress(round(i * 100 / request.frames), "Animating demo video…")\n                t = i / max(1, request.frames - 1)\n                frame = start.copy() if start else Image.new("RGB", (w, h), (int(40 + 60 * t), 70, 90))\n                draw = ImageDraw.Draw(frame)\n                r = int(min(w, h) * .12)\n                x = int(r + (w - 2 * r) * t); y = int(h / 2 + math.sin(t * math.pi * 4) * h * .2)\n                draw.ellipse((x - r, y - r, x + r, y + r), fill=(228, 236, 182))\n                draw.text((int(w * .04), int(h * .9)), "DEMO VIDEO - NOT AI", fill="#d2e6b4", font_size=max(12, w // 40))\n                process.stdin.write(frame.tobytes())\n            process.stdin.close()\n            if process.wait(timeout=120) or not temporary.is_file():\n                raise RuntimeError("Demo video encoding failed.")\n            temporary.replace(output)\n        finally:\n            if process.poll() is None:\n                process.kill(); process.wait()\n            temporary.unlink(missing_ok=True)\n', 'backend/schema.py': 'import base64\nimport binascii\nimport io\nfrom typing import Literal\nfrom PIL import Image\nfrom pydantic import BaseModel, Field, field_validator, model_validator\n\n\ndef validate_image_data(value: str, message: str) -> tuple[int, int]:\n    """Accept only an embedded PNG/JPEG/WebP data URL within the service\'s size limits."""\n    try:\n        if len(value) > 12 * 1024 * 1024:\n            raise ValueError()\n        header, encoded = value.split(\',\', 1)\n        if header not in (\'data:image/png;base64\', \'data:image/jpeg;base64\', \'data:image/webp;base64\'):\n            raise ValueError()\n        data = base64.b64decode(encoded, validate=True)\n        if len(data) > 8 * 1024 * 1024:\n            raise ValueError()\n        with Image.open(io.BytesIO(data)) as image:\n            if image.format not in (\'PNG\', \'JPEG\', \'WEBP\') or max(image.size) > 4096 or image.width * image.height > 16_000_000:\n                raise ValueError()\n            size = image.size\n            image.verify()\n        return size\n    except (ValueError, OSError, SyntaxError, binascii.Error, Image.DecompressionBombError):\n        raise ValueError(message) from None\n\n\ndef decode_image_data(value: str) -> Image.Image:\n    """Decode a data URL already accepted by validate_image_data."""\n    image = Image.open(io.BytesIO(base64.b64decode(value.split(\',\', 1)[1])))\n    image.load()\n    return image\n\n\nclass GenerateRequest(BaseModel):\n    prompt: str = Field(min_length=1, max_length=20000)\n    negative_prompt: str = Field(default="", max_length=20000)\n    width: int = Field(default=1024, ge=256, le=2048)\n    height: int = Field(default=1024, ge=256, le=2048)\n    steps: int = Field(default=28, ge=1, le=100)\n    guidance: float = Field(default=6, ge=0, le=20)\n    seed: int = Field(default=-1, ge=-1, le=2147483647)\n    reference_images: list[str] = Field(default_factory=list, max_length=3)\n    transparent: bool = False  # RGBA output with a transparent background (Qwen-Image 2.1)\n\n    @field_validator("reference_images")\n    @classmethod\n    def valid_references(cls, values):\n        for value in values:\n            validate_image_data(value, "Use a valid PNG, JPEG or WebP reference under 8 MB and 4096 pixels per side")\n        return values\n\n    @field_validator("prompt")\n    @classmethod\n    def nonempty(cls, value):\n        if not value.strip():\n            raise ValueError("Prompt cannot be blank")\n        return value.strip()\n\n    @field_validator("width", "height")\n    @classmethod\n    def alignment(cls, value):\n        if value % 32:\n            raise ValueError("Dimensions must be divisible by 32")\n        return value\n\n\nclass RemoveBackgroundRequest(BaseModel):\n    """One image for salient-subject matting. The result is a grayscale mask of the same size."""\n    image: str\n\n    @field_validator("image")\n    @classmethod\n    def valid_image(cls, value):\n        validate_image_data(value, "Send a valid PNG, JPEG or WebP image under 8 MB and 4096 pixels per side")\n        return value\n\n\nEDIT_OPERATIONS = ("edit", "inpaint", "outpaint", "image-to-image", "variations")\n\n\nclass EditRequest(GenerateRequest):\n    """Edit one image. The studio sends the source (and mask) already sized to width × height.\n\n    edit: instruction edit of the whole image (reference conditioning; needs the vision encoder).\n    inpaint / outpaint: regenerate the white area of `mask`, keep the black area.\n    image-to-image / variations: re-imagine the whole image; `strength` sets how far it may move.\n    """\n    operation: Literal["edit", "inpaint", "outpaint", "image-to-image", "variations"]\n    image: str\n    mask: str | None = None\n    strength: float | None = Field(default=None, ge=0.05, le=1)\n\n    @field_validator("image")\n    @classmethod\n    def valid_image(cls, value):\n        validate_image_data(value, "Send a valid PNG, JPEG or WebP source image under 8 MB and 4096 pixels per side")\n        return value\n\n    @field_validator("mask")\n    @classmethod\n    def valid_mask(cls, value):\n        if value is not None:\n            validate_image_data(value, "Send a valid PNG mask under 8 MB and 4096 pixels per side")\n        return value\n\n    @model_validator(mode="after")\n    def consistent(self):\n        if self.operation in ("inpaint", "outpaint") and not self.mask:\n            raise ValueError(f"{self.operation} needs a mask (white = area to regenerate)")\n        if len(self.reference_images) > 2:\n            raise ValueError("Use up to two extra reference images with an edit")\n        for value, label in ((self.image, "source image"), (self.mask, "mask")):\n            if value is not None and validate_image_data(value, "invalid") != (self.width, self.height):\n                raise ValueError(f"The {label} must be exactly {self.width} × {self.height} pixels")\n        return self\n\n    @property\n    def effective_strength(self) -> float:\n        if self.strength is not None:\n            return self.strength\n        return {"image-to-image": .6, "variations": .45}.get(self.operation, 1.0)\n\n\nclass VideoRequest(BaseModel):\n    """Text- or image-to-video. Frames follow the 8n + 1 rule of the LTX video VAE."""\n    prompt: str = Field(min_length=1, max_length=20000)\n    negative_prompt: str = Field(default="worst quality, low quality, blurry, distorted, artifacts", max_length=20000)\n    width: int = Field(default=768, ge=256, le=1920)\n    height: int = Field(default=512, ge=256, le=1920)\n    frames: int = Field(default=97, ge=9, le=257)\n    fps: int = Field(default=24, ge=8, le=50)\n    steps: int = Field(default=8, ge=1, le=100)\n    guidance: float = Field(default=1, ge=0, le=20)\n    seed: int = Field(default=-1, ge=-1, le=2147483647)\n    image: str | None = None\n    end_image: str | None = None\n\n    @field_validator("prompt")\n    @classmethod\n    def nonempty(cls, value):\n        if not value.strip():\n            raise ValueError("Prompt cannot be blank")\n        return value.strip()\n\n    @field_validator("width", "height")\n    @classmethod\n    def alignment(cls, value):\n        if value % 32:\n            raise ValueError("Video dimensions must be divisible by 32")\n        return value\n\n    @field_validator("frames")\n    @classmethod\n    def frame_rule(cls, value):\n        if (value - 1) % 8:\n            raise ValueError("Frame count must be 8n + 1 (for example 49, 97 or 121)")\n        return value\n\n    @field_validator("image", "end_image")\n    @classmethod\n    def valid_frame(cls, value):\n        if value is not None:\n            validate_image_data(value, "Send a valid PNG, JPEG or WebP frame under 8 MB and 4096 pixels per side")\n        return value\n\n\n# Audio and language ---------------------------------------------------------------------\nAUDIO_TYPES = ("audio/wav", "audio/x-wav", "audio/wave", "audio/mpeg", "audio/mp3", "audio/mp4", "audio/x-m4a",\n               "audio/aac", "audio/ogg", "audio/webm", "audio/flac", "audio/x-flac", "video/mp4", "video/webm",\n               "video/quicktime")\nAUDIO_LIMIT = 32 * 1024 * 1024\n\n\ndef validate_audio_data(value: str, message: str) -> int:\n    """Accept an embedded audio (or video with sound) data URL; returns its decoded size."""\n    try:\n        if len(value) > AUDIO_LIMIT * 4 // 3 + 128:\n            raise ValueError()\n        header, encoded = value.split(",", 1)\n        if not header.endswith(";base64"):\n            raise ValueError()\n        media = header[5:-7].split(";")[0].lower()\n        if not header.startswith("data:") or media not in AUDIO_TYPES:\n            raise ValueError()\n        data = base64.b64decode(encoded, validate=True)\n        if not 64 <= len(data) <= AUDIO_LIMIT:\n            raise ValueError()\n        return len(data)\n    except (ValueError, binascii.Error):\n        raise ValueError(message) from None\n\n\ndef decode_audio_data(value: str) -> bytes:\n    return base64.b64decode(value.split(",", 1)[1])\n\n\ndef _text(value: str, label: str) -> str:\n    if not value.strip():\n        raise ValueError(f"{label} cannot be blank")\n    return value.strip()\n\n\nclass TranscribeRequest(BaseModel):\n    """Audio or a video with sound; the result is text with timed segments."""\n    audio: str\n    language: str | None = Field(default=None, min_length=2, max_length=5, pattern=r"^[a-z]{2,3}(-[a-z]{2})?$")\n\n    @field_validator("audio")\n    @classmethod\n    def valid_audio(cls, value):\n        validate_audio_data(value, "Send a WAV, MP3, M4A, OGG, WebM, FLAC or MP4 file under 32 MB")\n        return value\n\n\nclass DescribeRequest(BaseModel):\n    """Describe an image as a prompt (to recreate or animate it) or as a plain caption."""\n    image: str\n    purpose: Literal["image", "video", "caption"] = "image"\n\n    @field_validator("image")\n    @classmethod\n    def valid_image(cls, value):\n        validate_image_data(value, "Send a valid PNG, JPEG or WebP image under 8 MB and 4096 pixels per side")\n        return value\n\n\nclass EnhancePromptRequest(BaseModel):\n    """Rewrite a short idea into a detailed prompt for one of the studio\'s generators."""\n    prompt: str = Field(min_length=1, max_length=4000)\n    target: Literal["image", "edit", "video", "music"] = "image"\n\n    @field_validator("prompt")\n    @classmethod\n    def nonempty(cls, value):\n        return _text(value, "Prompt")\n\n\nclass SpeechRequest(BaseModel):\n    """Text to speech. `voice` is an optional reference clip (over 5 seconds) whose voice is cloned."""\n    text: str = Field(min_length=1, max_length=5000)\n    voice: str | None = None\n    temperature: float = Field(default=0.8, ge=0.05, le=2)\n    seed: int = Field(default=-1, ge=-1, le=2147483647)\n\n    @field_validator("text")\n    @classmethod\n    def nonempty(cls, value):\n        return _text(value, "Text")\n\n    @field_validator("voice")\n    @classmethod\n    def valid_voice(cls, value):\n        if value is not None:\n            validate_audio_data(value, "Send the voice sample as a WAV, MP3, M4A, OGG, WebM or FLAC file under 32 MB")\n        return value\n\n\nclass MusicRequest(BaseModel):\n    """A song from a style description and optional lyrics with [verse]/[chorus] tags on their own lines."""\n    style: str = Field(min_length=1, max_length=2000)\n    lyrics: str = Field(default="", max_length=3500)\n    duration: int = Field(default=60, ge=10, le=300)\n    instrumental: bool = False\n    seed: int = Field(default=-1, ge=-1, le=2147483647)\n\n    @field_validator("style")\n    @classmethod\n    def nonempty(cls, value):\n        return _text(value, "Style")\n\n    @model_validator(mode="after")\n    def lyrics_or_instrumental(self):\n        if not self.instrumental and not self.lyrics.strip():\n            raise ValueError("Add lyrics, or choose instrumental")\n        return self\n\n\nclass UpscaleRequest(BaseModel):\n    """Upscale one image ×2 or ×4 (output up to 8192 pixels per side)."""\n    image: str\n    scale: Literal[2, 4] = 2\n\n    @field_validator("image")\n    @classmethod\n    def valid_image(cls, value):\n        validate_image_data(value, "Send a valid PNG, JPEG or WebP image under 8 MB and 4096 pixels per side")\n        return value\n\n    @property\n    def source_size(self) -> tuple[int, int]:\n        return validate_image_data(self.image, "invalid")\n\n\nclass DetectPoseRequest(BaseModel):\n    """Find people and their body pose (OpenPose-18 keypoints) in one image."""\n    image: str\n\n    @field_validator("image")\n    @classmethod\n    def valid_image(cls, value):\n        validate_image_data(value, "Send a valid PNG, JPEG or WebP image under 8 MB and 4096 pixels per side")\n        return value\n', 'backend/vision.py': '"""Upscaling and pose detection.\n\nUpscaler      Real-ESRGAN x4plus (BSD-3) through spandrel, tiled so any image size fits in memory.\n              ×2 runs the ×4 model and downsamples with Lanczos (sharper than a native ×2 model).\nPoseDetector  DWPose whole-body (Apache-2.0) with a YOLOX person detector through rtmlib on ONNX Runtime;\n              returns OpenPose-18 body keypoints that the studio\'s pose editor understands.\n\nBoth load on first use (small: 67 MB and ~240 MB) and follow ZERO_HELPERS_RESIDENT. Demo twins need\nno weights so the studio and its tests run anywhere.\n"""\nimport os\nfrom pathlib import Path\nfrom threading import RLock\n\nfrom PIL import Image, ImageDraw\n\nfrom .audio import free_torch, helpers_resident\nfrom .runner import Cancelled\n\nUPSCALE_MODEL = "Comfy-Org/Real-ESRGAN_repackaged/RealESRGAN_x4plus"\nPOSE_MODEL = "yzd-v/DWPose/dw-ll_ucoco_384 + hr16/yolox-onnx/yolox_m"\nMAX_OUTPUT_EDGE = 8192\nMAX_OUTPUT_PIXELS = 48_000_000\n\n# OpenPose-18 body: 0 nose, 1 neck, 2-4 right arm, 5-7 left arm, 8-10 right leg, 11-13 left leg, 14-17 eyes/ears.\nLIMBS = [(1, 2), (1, 5), (2, 3), (3, 4), (5, 6), (6, 7), (1, 8), (8, 9), (9, 10), (1, 11), (11, 12), (12, 13),\n         (1, 0), (0, 14), (14, 16), (0, 15), (15, 17)]\nCOLORS = [(255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0), (85, 255, 0), (0, 255, 0),\n          (0, 255, 85), (0, 255, 170), (0, 255, 255), (0, 170, 255), (0, 85, 255), (0, 0, 255), (85, 0, 255),\n          (170, 0, 255), (255, 0, 255), (255, 0, 170), (255, 0, 85)]\n\n\ndef upscaled_size(width: int, height: int, scale: int):\n    return width * scale, height * scale\n\n\ndef check_upscale_size(width: int, height: int, scale: int):\n    w, h = upscaled_size(width, height, scale)\n    if max(w, h) > MAX_OUTPUT_EDGE or w * h > MAX_OUTPUT_PIXELS:\n        raise ValueError(f"×{scale} would make {w} × {h} pixels; the limit is {MAX_OUTPUT_EDGE} pixels per side "\n                         f"and {MAX_OUTPUT_PIXELS // 1_000_000} megapixels. Use ×2 or a smaller image.")\n\n\ndef split_alpha(image: Image.Image):\n    """RGB for the model; the alpha channel (if any) is resized separately so cut-outs stay cut out."""\n    if image.mode in ("RGBA", "LA") or (image.mode == "P" and "transparency" in image.info):\n        rgba = image.convert("RGBA")\n        alpha = rgba.getchannel("A")\n        return rgba.convert("RGB"), (alpha if alpha.getextrema() != (255, 255) else None)\n    return image.convert("RGB"), None\n\n\nclass Upscaler:\n    mode = "qwen"\n    label = "upscale"\n    runner_name = "spandrel"\n\n    def __init__(self, path=None):\n        self.path = path or os.environ.get("ZERO_UPSCALER", "")\n        self.model = os.environ.get("ZERO_UPSCALER_ID", UPSCALE_MODEL)\n        self.lock = RLock()\n        self.net = None\n\n    def missing(self):\n        return [] if self.path and Path(self.path).is_file() else ["upscaling model"]\n\n    def _load(self):\n        import torch\n        from spandrel import ImageModelDescriptor, ModelLoader\n        net = ModelLoader().load_from_file(self.path)\n        if not isinstance(net, ImageModelDescriptor):\n            raise RuntimeError("The upscaling file is not an image model.")\n        device = "cuda" if torch.cuda.is_available() else "cpu"\n        net.to(device).eval()\n        if device == "cuda":\n            net.half() if net.supports_half else net.bfloat16()\n        return net\n\n    def _tiled(self, net, image, cancelled, progress, tile=512, overlap=32):\n        """Run the model in overlapping tiles with linear blending (seamless, bounded memory)."""\n        import numpy as np\n        import torch\n        s = net.scale\n        param = next(net.model.parameters())\n        x = torch.from_numpy(np.asarray(image, dtype=np.uint8).copy()).permute(2, 0, 1)[None].to(param.device, param.dtype) / 255\n        _, _, H, W = x.shape\n        out = torch.zeros((1, 3, H * s, W * s), device=param.device)\n        weight = torch.zeros_like(out[:, :1])\n        step = tile - overlap\n        ys = list(range(0, max(H - overlap, 1), step))\n        xs = list(range(0, max(W - overlap, 1), step))\n        done, total = 0, len(ys) * len(xs)\n        with torch.inference_mode():\n            for y0 in ys:\n                for x0 in xs:\n                    if cancelled.is_set():\n                        raise Cancelled()\n                    y1, x1 = min(y0 + tile, H), min(x0 + tile, W)\n                    y0_, x0_ = max(y1 - tile, 0), max(x1 - tile, 0)\n                    t = net(x[:, :, y0_:y1, x0_:x1]).float()\n                    th, tw = t.shape[-2:]\n                    ry = torch.ones(th, device=param.device)\n                    rx = torch.ones(tw, device=param.device)\n                    f = overlap * s\n                    ramp = torch.linspace(0, 1, f + 2, device=param.device)[1:-1]\n                    if y0_ > 0: ry[:f] = ramp\n                    if y1 < H: ry[-f:] = ramp.flip(0)\n                    if x0_ > 0: rx[:f] = ramp\n                    if x1 < W: rx[-f:] = ramp.flip(0)\n                    m = (ry[:, None] * rx[None])[None, None]\n                    out[:, :, y0_ * s:y1 * s, x0_ * s:x1 * s] += t * m\n                    weight[:, :, y0_ * s:y1 * s, x0_ * s:x1 * s] += m\n                    done += 1\n                    progress(round(done * 100 / total), f"Upscaling tile {done} of {total}…")\n        result = (out / weight.clamp_min(1e-6)).clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy()\n        return Image.fromarray((result * 255).round().astype("uint8"))\n\n    def upscale(self, request, image: Image.Image, output: Path, cancelled, progress):\n        check_upscale_size(image.width, image.height, request.scale)\n        rgb, alpha = split_alpha(image)\n        with self.lock:\n            if self.net is None:\n                progress(None, "Loading the upscaling model…")\n                self.net = self._load()\n            big = self._tiled(self.net, rgb, cancelled, progress)\n            if not helpers_resident():\n                self.net = None\n                free_torch()\n        size = upscaled_size(image.width, image.height, request.scale)\n        if big.size != size:\n            big = big.resize(size, Image.LANCZOS)\n        if alpha is not None:\n            big = big.convert("RGBA")\n            big.putalpha(alpha.resize(size, Image.LANCZOS))\n        big.save(output)\n\n\nclass DemoUpscaler:\n    mode = "demo"\n    label = "upscale"\n    model = "studio-zero/demo-upscale"\n    runner_name = "procedural"\n\n    def missing(self):\n        return []\n\n    def upscale(self, request, image: Image.Image, output: Path, cancelled, progress):\n        check_upscale_size(image.width, image.height, request.scale)\n        big = image.convert("RGBA").resize(upscaled_size(image.width, image.height, request.scale), Image.LANCZOS)\n        ImageDraw.Draw(big).text((8, 8), "DEMO UPSCALE - NOT AI", fill=(255, 64, 64, 255))\n        big.save(output)\n\n\n# Pose -------------------------------------------------------------------------------------\ndef draw_pose(people, width: int, height: int, threshold=0.3) -> Image.Image:\n    """The standard OpenPose-18 skeleton picture (black background) for a list of keypoint lists."""\n    canvas = Image.new("RGB", (width, height), (0, 0, 0))\n    draw = ImageDraw.Draw(canvas)\n    stick = max(2, round(min(width, height) / 160))\n    for person in people:\n        for i, (a, b) in enumerate(LIMBS):\n            pa, pb = person[a], person[b]\n            if pa[2] < threshold or pb[2] < threshold:\n                continue\n            color = tuple(int(c * 0.6) for c in COLORS[i])\n            draw.line([(pa[0], pa[1]), (pb[0], pb[1])], fill=color, width=stick * 2)\n        for i, (x, y, c) in enumerate(person[:18]):\n            if c >= threshold:\n                draw.ellipse([x - stick, y - stick, x + stick, y + stick], fill=COLORS[i])\n    return canvas\n\n\nclass PoseDetector:\n    mode = "qwen"\n    label = "pose"\n\n    def __init__(self, pose_path=None, detector_path=None):\n        self.pose_path = pose_path or os.environ.get("ZERO_POSE_MODEL", "")\n        self.detector_path = detector_path or os.environ.get("ZERO_POSE_DETECTOR", "")\n        self.model = os.environ.get("ZERO_POSE_MODEL_ID", POSE_MODEL)\n        self.lock = RLock()\n        self.models = None\n        self.device = None\n\n    def missing(self):\n        ok = all(p and Path(p).is_file() for p in (self.pose_path, self.detector_path))\n        return [] if ok else ["pose model"]\n\n    def _load(self):\n        import onnxruntime as ort\n        from rtmlib import YOLOX, RTMPose\n        if hasattr(ort, "preload_dlls"):\n            try:\n                ort.preload_dlls()\n            except Exception:\n                pass\n        device = "cuda" if "CUDAExecutionProvider" in ort.get_available_providers() else "cpu"\n        # det_mode="multiclass" returns class ids too: the COCO detector would otherwise put skeletons on chairs.\n        detector = YOLOX(self.detector_path, model_input_size=(640, 640), det_mode="multiclass", score_thr=0.5,\n                         backend="onnxruntime", device=device)\n        pose = RTMPose(self.pose_path, model_input_size=(288, 384), to_openpose=True, backend="onnxruntime", device=device)\n        return (detector, pose), device\n\n    def detect(self, image: Image.Image) -> dict:\n        import numpy as np\n        rgb = image.convert("RGB")\n        bgr = np.ascontiguousarray(np.asarray(rgb)[:, :, ::-1])\n        with self.lock:\n            if self.models is None:\n                self.models, self.device = self._load()\n            detector, pose = self.models\n            found = detector(bgr)\n            boxes, classes = (found if isinstance(found, tuple) else (found, None))\n            boxes = np.asarray(boxes).reshape(-1, 4)\n            if classes is not None:\n                boxes = boxes[np.asarray(classes).reshape(-1) == 0]  # people only (COCO class 0)\n            people = []\n            if len(boxes):\n                keypoints, scores = pose(bgr, bboxes=boxes)\n                for k, s in zip(np.asarray(keypoints)[:, :18], np.asarray(scores)[:, :18]):\n                    people.append([[round(float(x), 1), round(float(y), 1), round(float(c), 3)] for (x, y), c in zip(k, s)])\n            if not helpers_resident():\n                self.models = None\n        people.sort(key=lambda p: -sum(c for _, _, c in p))\n        return {"width": rgb.width, "height": rgb.height, "people": people[:6], "model": self.model, "device": self.device}\n\n\n# A standing figure in a 1 × 2 box (x, y as fractions), used by the demo detector and the studio\'s presets.\nSTANDING = [(0.5, 0.11), (0.5, 0.2), (0.38, 0.21), (0.33, 0.36), (0.31, 0.5), (0.62, 0.21), (0.67, 0.36),\n            (0.69, 0.5), (0.43, 0.5), (0.43, 0.7), (0.43, 0.9), (0.57, 0.5), (0.57, 0.7), (0.57, 0.9),\n            (0.48, 0.095), (0.52, 0.095), (0.45, 0.105), (0.55, 0.105)]\n\n\nclass DemoPoseDetector:\n    mode = "demo"\n    label = "pose"\n    model = "studio-zero/demo-pose"\n    device = "cpu"\n\n    def missing(self):\n        return []\n\n    def detect(self, image: Image.Image) -> dict:\n        w, h = image.size\n        box_h = h * 0.9\n        box_w = box_h / 2\n        left, top = (w - box_w) / 2, h * 0.05\n        person = [[round(left + x * box_w, 1), round(top + y * box_h, 1), 0.9] for x, y in STANDING]\n        return {"width": w, "height": h, "people": [person], "model": self.model, "device": self.device}\n'}
for name, content in SOURCES.items():
    target = ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
sys.path.insert(0, str(ROOT))
ok(f"Studio Zero API installed ({len(SOURCES)} files)")


In [ ]:
#@title 3 · Get the models (from Drive; downloads only what is missing) { display-mode: "form" }
from huggingface_hub import HfApi, hf_hub_download
from backend.runner import SD_REVISION
from backend.model_cache import ModelFile, ModelStore, RepoSnapshot, link_view
os.environ['HF_XET_HIGH_PERFORMANCE'] = '1'
def colab_secret(name):
    """Optional Colab secrets (HF_TOKEN, DEEPSEEK_API_KEY) stay on this runtime."""
    if not os.environ.get(name):
        try:
            from google.colab import userdata
            os.environ[name] = userdata.get(name)
        except Exception:
            os.environ.pop(name, None)
# Hugging Face token for gated repositories (LTX-2.5): the Settings field wins over the Colab secret.
os.environ.pop('HF_TOKEN', None)
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN.strip()
else:
    colab_secret('HF_TOKEN')
if os.environ.get('HF_TOKEN'):
    try:
        account = HfApi().whoami(token=os.environ['HF_TOKEN'])
        ok(f"Hugging Face token accepted ({account.get('name', 'your account')})")
    except Exception as exc:
        if getattr(getattr(exc, 'response', None), 'status_code', None) == 401:
            os.environ.pop('HF_TOKEN')  # a bad token would also block public downloads
            print('• The Hugging Face token was rejected, so it is not used. Check HF_TOKEN in Settings.')
        else:
            print(f'• Could not check the Hugging Face token right now ({type(exc).__name__}); using it anyway.')
colab_secret('DEEPSEEK_API_KEY')  # Improve prompt with DeepSeek's API instead of the local model

# Which files are needed -----------------------------------------------------------------
ENCODER_REPO = 'Qwen/Qwen3-VL-8B-Instruct-GGUF'
hub = HfApi()
def resolve(repo, revision):
    info = hub.model_info(repo, revision=revision, files_metadata=True)
    return RepoSnapshot(info.sha, {s.rfilename: s.size for s in info.siblings})
def repo_files(repo, keep):
    """A repository's file list, remembered on Drive so later sessions need no network."""
    listing = (CACHE / 'models' / repo / '.studio-zero-files.json') if CACHE else None
    if listing and listing.is_file() and not (UPDATE_MODELS or REDOWNLOAD_MODELS):
        names = json.loads(listing.read_text())
    else:
        names = sorted(resolve(repo, 'main').sizes)
        if listing:
            listing.parent.mkdir(parents=True, exist_ok=True); listing.write_text(json.dumps(names))
    return [name for name in names if keep(name)]
files = [ModelFile('ZERO_DIFFUSION', DIFFUSION_REPO, DIFFUSION_FILE)]
for env in ('ZERO_VAE', 'ZERO_TEXT_ENCODER', 'ZERO_VISION_ENCODER', 'ZERO_MATTING_MODEL'):
    os.environ.pop(env, None)
if FAST:
    # The UC GGUF is the transformer; the companion supplies text encoder, VAE, processor and configs.
    files += [ModelFile('', COMPANION_REPO, name) for name in repo_files(
        COMPANION_REPO, lambda n: not n.startswith(('assets/', '.')) and not (n.startswith('transformer/') and n.endswith('.safetensors')))]
    files += [ModelFile('', FAST_REPO, name) for name in (FAST_FILE, FAST_FILE.rsplit('/', 1)[0] + '/pdd_config.json',
                                                           'qwenimage21_pdd.py', 'lora_utils_pdd.py')]
else:
    files += [ModelFile('ZERO_VAE', DIFFUSION_REPO, 'vae/qwen_image_2.1_vae_bf16.safetensors'),
              ModelFile('ZERO_TEXT_ENCODER', ENCODER_REPO, 'Qwen3VL-8B-Instruct-Q4_K_M.gguf')]
    if ENABLE_REFERENCES:
        files.append(ModelFile('ZERO_VISION_ENCODER', ENCODER_REPO, 'mmproj-Qwen3VL-8B-Instruct-F16.gguf'))
if ENABLE_BACKGROUND_REMOVAL:
    files.append(ModelFile('ZERO_MATTING_MODEL', MATTING_REPO, MATTING_FILE))
video_env = {'ZERO_VIDEO_DIFFUSION': VIDEO_DIFFUSION, 'ZERO_VIDEO_TEXT_ENCODER': VIDEO_TEXT_ENCODER,
             'ZERO_VIDEO_VAE': VIDEO_VAE, 'ZERO_VIDEO_AUDIO_VAE': VIDEO_AUDIO_VAE}
for env in video_env:
    os.environ.pop(env, None)
if ENABLE_VIDEO:
    gated = VIDEO_TEXT_ENCODER[0]
    if not os.environ.get('HF_TOKEN'):
        print('• No Hugging Face token: set HF_TOKEN in Settings (or the Colab secret) or the gated video files will not download.')
    else:
        try:
            from huggingface_hub import auth_check
            auth_check(gated, token=os.environ['HF_TOKEN'])
        except ImportError:
            pass
        except Exception:
            print(f'• Your token cannot open {gated} yet: accept the licence at https://huggingface.co/{gated}, then run this step again.')
    files += [ModelFile(env, repo, filename) for env, (repo, filename) in video_env.items()]
MUSIC_PARTS = ('language_model/', 'transformer/', 'rvq_depth_decoder/', 'vocoder/', 'condition_encoder/', 'tokenizer/', 'scheduler/')
helper_repos = {  # env: (enabled, repo, files to keep)
    'ZERO_VL': (ENABLE_ASSISTANT, ASSISTANT_REPO, lambda n: not n.startswith('.') and n != 'README.md'),
    'ZERO_WHISPER': (ENABLE_TRANSCRIBE, WHISPER_REPO, lambda n: not n.startswith('.') and n != 'README.md'),
    'ZERO_TTS': (ENABLE_VOICE, VOICE_REPO, lambda n: not n.startswith('.') and n not in ('README.md', 's3gen.safetensors')),
    'ZERO_MUSIC': (ENABLE_MUSIC, MUSIC_REPO, lambda n: n.startswith(MUSIC_PARTS) or n in ('modular_model_index.json', 'config.json', 'LICENSE')),
}
for env, (enabled, repo, keep) in helper_repos.items():
    os.environ.pop(env, None)
    if enabled:
        files += [ModelFile('', repo, name) for name in repo_files(repo, keep)]
vision_env = {'ZERO_UPSCALER': (ENABLE_UPSCALE, UPSCALE_MODEL), 'ZERO_POSE_MODEL': (ENABLE_POSE, POSE_MODEL),
              'ZERO_POSE_DETECTOR': (ENABLE_POSE, POSE_DETECTOR)}
for env, (enabled, (repo, filename)) in vision_env.items():
    os.environ.pop(env, None)
    if enabled:
        files.append(ModelFile(env, repo, filename))

# Weights: reuse the Drive cache; download (in a background thread) only what is missing -----
def download(repo, filename, revision, destination):
    return hf_hub_download(repo_id=repo, filename=filename, revision=revision, local_dir=str(destination),
                           force_download=REDOWNLOAD_MODELS)
model_store = ModelStore(ROOT / 'models', CACHE, update=UPDATE_MODELS, redownload=REDOWNLOAD_MODELS,
                         copy_to_local=COPY_WEIGHTS_TO_LOCAL)
legacy = (lambda item: CACHE / 'models' / item.repo.split('/')[0] / item.filename) if CACHE else None
download_error = []
def prepare_weights():
    try:
        os.environ.update(model_store.prepare(files, resolve, download, legacy=legacy))
    except Exception as exc:
        download_error.append(exc)
t0 = time.monotonic()
downloader = threading.Thread(target=prepare_weights, daemon=True); downloader.start()

# Engine: restore a cached stable-diffusion.cpp build for this revision / GPU / CUDA, or build it once.
# The fast engine runs in Python; the compiled engine is still needed for video.
NEED_SDCPP = not FAST or ENABLE_VIDEO
cuda_version = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout.rsplit('release ', 1)[-1].split(',')[0]
engine_key = f'{SD_REVISION[:12]}-sm{GPU_ARCH}-cuda{cuda_version}'
engine_dir = ROOT / 'engine' / engine_key
engine = engine_dir / 'sd-server'
required = ['--diffusion-model', '--llm', '--vae', '--offload-to-cpu', '--listen-ip', '--listen-port', '--diffusion-fa',
            '--sage-attn', '--audio-vae', '--vae-tiling']
def engine_works(path):
    try:
        text = subprocess.run([str(path), '--help'], capture_output=True, text=True, timeout=60).stdout
        return all(flag in text for flag in required)
    except (OSError, subprocess.SubprocessError):
        return False
if NEED_SDCPP:
    cached = (CACHE / 'engine' / engine_key / 'sd-server') if CACHE else None
    if REBUILD_ENGINE and 'engine_rebuilt' not in globals():
        shutil.rmtree(engine_dir, ignore_errors=True)
        shutil.rmtree('/content/stable-diffusion.cpp/build', ignore_errors=True)
        cached = None   # skip the Drive copy; the fresh build replaces it below
        engine_rebuilt = True
    if not engine.exists() and cached and cached.exists():
        engine_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cached, engine); engine.chmod(0o755)
        ok(f'Engine restored from Drive ({engine_key})')
    if not (engine.exists() and engine_works(engine)):
        src = Path('/content/stable-diffusion.cpp')
        if not (src / '.git').exists():
            subprocess.run(['git', 'init', '-q', str(src)], check=True)
            subprocess.run(['git', '-C', str(src), 'remote', 'add', 'origin', 'https://github.com/leejet/stable-diffusion.cpp.git'], check=True)
        subprocess.run(['git', '-C', str(src), 'fetch', '-q', '--depth', '1', 'origin', SD_REVISION], check=True)
        subprocess.run(['git', '-C', str(src), 'checkout', '-q', 'FETCH_HEAD'], check=True)
        subprocess.run(['git', '-C', str(src), 'submodule', 'update', '-q', '--init', '--recursive', '--depth', '1'], check=True)
        subprocess.run(['cmake', '-S', str(src), '-B', str(src / 'build'), '-DCMAKE_BUILD_TYPE=Release',
                        '-DSD_CUDA=ON', f'-DCMAKE_CUDA_ARCHITECTURES={GPU_ARCH}', '-DSD_WEBP=OFF', '-DSD_WEBM=OFF',
                        '-DSD_SERVER_BUILD_FRONTEND=OFF'], check=True, stdout=subprocess.DEVNULL)
        print(f'• Building the engine for sm_{GPU_ARCH} on {os.cpu_count()} cores (first session only)…')
        subprocess.run(['cmake', '--build', str(src / 'build'), '--config', 'Release', '-j', str(os.cpu_count()), '--target', 'sd-server'],
                       check=True, stdout=subprocess.DEVNULL)
        engine_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src / 'build' / 'bin' / 'sd-server', engine); engine.chmod(0o755)
        assert engine_works(engine), 'The built engine is missing a required option.'
        drive_engine = (CACHE / 'engine' / engine_key / 'sd-server') if CACHE else None
        if drive_engine:
            drive_engine.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(engine, drive_engine)
        ok('Engine built' + (' and saved to Drive' if drive_engine else ''))
os.environ['ZERO_SD_SERVER'] = str(engine) if NEED_SDCPP else ''
os.environ['ZERO_SD_REVISION'] = SD_REVISION

downloader.join()
if download_error:
    raise download_error[0]
revisions = model_store.revisions
def model_id(repo, filename):
    return f"{repo}/{filename.rsplit('.', 1)[0]}@{revisions[repo][:12]}"
os.environ['ZERO_MODEL_ID'] = model_id(DIFFUSION_REPO, DIFFUSION_FILE)
if ENABLE_VIDEO:
    os.environ['ZERO_VIDEO_MODEL_ID'] = model_id(*VIDEO_DIFFUSION)
if ENABLE_BACKGROUND_REMOVAL:
    os.environ['ZERO_MATTING_MODEL_ID'] = f'{MATTING_REPO}/{MATTING_FILE}@{revisions[MATTING_REPO][:12]}'
if FAST:
    companion = link_view(model_store.paths, COMPANION_REPO, ROOT / 'views' / 'companion')
    fun_acc = link_view(model_store.paths, FAST_REPO, ROOT / 'views' / 'fun-acc')
    os.environ.update(ZERO_DIFFUSERS_COMPANION=str(companion), ZERO_FAST_LORA=str(fun_acc / FAST_FILE),
                      ZERO_FAST_CODE=str(fun_acc))
    os.environ['ZERO_MODEL_ID'] += ' · diffusers + Fun-Acc 4-step'
for env, (enabled, repo, keep) in helper_repos.items():
    if enabled:
        os.environ[env] = str(link_view(model_store.paths, repo, ROOT / 'views' / env.lower()))
        os.environ[env + '_ID'] = f'{repo}@{revisions[repo][:12]}'
if ENABLE_UPSCALE:
    os.environ['ZERO_UPSCALER_ID'] = model_id(*UPSCALE_MODEL)
if ENABLE_POSE:
    os.environ['ZERO_POSE_MODEL_ID'] = f"{POSE_MODEL[0]}/{POSE_MODEL[1]} + {POSE_DETECTOR[0]}/{POSE_DETECTOR[1]}"
(ROOT / 'model-revisions.json').write_text(json.dumps(revisions, indent=2))
ok(f'Models ready in {time.monotonic() - t0:.0f}s · {model_store.summary()}')


In [ ]:
#@title 4 · Start the engine (loads the image model into GPU memory) { display-mode: "form" }
import httpx
if 'server_process' in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=30)
if os.environ.get('ZERO_SD_SERVER'):
    subprocess.run(['pkill', '-f', os.environ['ZERO_SD_SERVER']])  # clear an engine orphaned by a hard stop
os.environ.update({
    'ZERO_API_TOKEN': secrets.token_urlsafe(32),
    'ZERO_RUNNER': 'diffusers' if FAST else 'qwen',
    'ZERO_FLASH_ATTENTION': '1' if FLASH_ATTENTION else '0',
    'ZERO_SAGE_ATTENTION': '1' if SAGE_ATTENTION else '0',
    'ZERO_OFFLOAD': str(OFFLOAD),
    'ZERO_ENGINE_SWAP': '1' if ENGINE_SWAP else '0',
    'ZERO_VIDEO_FORMAT': VIDEO_FORMAT,
    'ZERO_ALLOWED_ORIGINS': 'http://localhost:5173,http://127.0.0.1:5173,http://localhost:4173,http://127.0.0.1:4173',
    'ZERO_OUTPUT_DIR': str(ROOT / 'outputs'),
    'ZERO_JOB_TIMEOUT': '3600',
})
service_log = (ROOT / 'service.log').open('w')
t0 = time.monotonic()
# The API binds to loopback only; step 5 adds an authenticated HTTPS tunnel.
server_process = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'backend.app:create_app', '--factory',
                                  '--host', '127.0.0.1', '--port', '8000', '--workers', '1'],
                                 cwd=ROOT, env=os.environ.copy(), stdout=service_log, stderr=subprocess.STDOUT)
api = httpx.Client(base_url='http://127.0.0.1:8000', timeout=30,
                   headers={'Authorization': 'Bearer ' + os.environ['ZERO_API_TOKEN']})
health, shown = None, None
while time.monotonic() - t0 < 1800:
    if server_process.poll() is not None:
        raise RuntimeError('The service stopped. See /content/studio-zero/service.log')
    try:
        health = api.get('/health').raise_for_status().json()
    except httpx.TransportError:
        time.sleep(0.5); continue
    if health['ready']:
        break
    if health.get('engine') == 'failed' or not health['message'].startswith('Loading'):
        raise RuntimeError(health['message'])
    if shown != health['message']:
        print('•', health['message']); shown = health['message']
    time.sleep(1)
else:
    raise RuntimeError('The engine did not load in time. See /content/studio-zero/outputs/engine.log')
where = 'offloaded to system RAM (slower)' if health.get('offload') else 'kept in GPU memory'
ok(f"Engine ready in {time.monotonic() - t0:.0f}s · weights {where}")
ok('Features: ' + ', '.join(health['capabilities']))

def generate(settings, path='/generate', timeout=None):
    """Submit a job and wait for it; returns (job, seconds)."""
    start = time.monotonic()
    job = api.post(path, json=settings, timeout=timeout or 30).raise_for_status().json()
    while job['status'] in ('queued', 'running'):
        if server_process.poll() is not None:
            raise RuntimeError('The service stopped. See /content/studio-zero/service.log')
        time.sleep(1)
        job = api.get('/jobs/' + job['id']).raise_for_status().json()
    assert job['status'] == 'succeeded', job['message']
    return job, time.monotonic() - start

def run_task(path, payload, timeout=600):
    """Submit a task and wait for it; returns (result, seconds)."""
    start = time.monotonic()
    task = api.post(path, json=payload, timeout=120).raise_for_status().json()
    while task['status'] in ('queued', 'running'):
        if time.monotonic() - start > timeout:
            raise RuntimeError(f'{path} took too long: ' + task['message'])
        time.sleep(0.5)
        task = api.get('/tasks/' + task['id']).raise_for_status().json()
    assert task['status'] == 'succeeded', task['message']
    return task['result'], time.monotonic() - start


## Connect
Cloudflare Quick Tunnel carries the traffic over HTTPS. Every request needs the access key, so the address alone
cannot use your engine. A new key is made each session. Clear this cell's output before sharing the notebook.

In [ ]:
#@title 5 · Connect your studio { display-mode: "form" }
import re, html, urllib.parse, urllib.request
from IPython.display import HTML, display
cloudflared = ROOT / 'cloudflared'
cached_tunnel = (CACHE / 'tools' / 'cloudflared') if CACHE else None
if not cloudflared.exists() and cached_tunnel and cached_tunnel.exists():
    shutil.copy2(cached_tunnel, cloudflared)
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    if cached_tunnel:
        cached_tunnel.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(cloudflared, cached_tunnel)
cloudflared.chmod(0o755)
if 'tunnel_process' in globals() and tunnel_process.poll() is None:
    tunnel_process.terminate(); tunnel_process.wait(timeout=15)
tunnel_log_path = ROOT / 'tunnel.log'
tunnel_log = tunnel_log_path.open('w')
tunnel_process = subprocess.Popen([str(cloudflared), 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                                  stdout=tunnel_log, stderr=subprocess.STDOUT)
for attempt in range(90):
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', tunnel_log_path.read_text())
    if match:
        break
    time.sleep(1)
else:
    raise RuntimeError('The tunnel did not start. See /content/studio-zero/tunnel.log')
SERVICE_URL, ACCESS_KEY = match.group(0), os.environ['ZERO_API_TOKEN']
# The link carries the details in the #fragment, which browsers never send to a server.
STUDIO_LINK = ('http://localhost:5173/#connect=' + urllib.parse.quote(SERVICE_URL, safe='') +
               '&key=' + urllib.parse.quote(ACCESS_KEY, safe=''))
field = ('width:100%;box-sizing:border-box;font:13px ui-monospace,monospace;padding:7px 9px;border-radius:8px;'
         'border:1px solid #8886;background:transparent;color:inherit')
def copy_row(label, value, ident):
    return (f'<div style="margin-top:10px"><div style="font-size:12px;opacity:.7;margin-bottom:4px">{label}</div>'
            f'<div style="display:flex;gap:6px"><input id="{ident}" readonly value="{html.escape(value)}" '
            f'onclick="this.select()" style="{field}"><button onclick="const f=document.getElementById(\'{ident}\');'
            f"f.select();(navigator.clipboard?navigator.clipboard.writeText(f.value):Promise.reject()).catch(()=>document.execCommand('copy'));"
            f"this.textContent='Copied'\" style=\"padding:6px 12px;border-radius:8px;border:1px solid #8886;"
            f'background:transparent;color:inherit;cursor:pointer">Copy</button></div></div>')
display(HTML(
    '<div style="font-family:system-ui,sans-serif;border:1px solid #8885;border-radius:14px;padding:16px 18px;max-width:640px">'
    '<div style="font-size:17px;font-weight:600">✓ Studio Zero is ready</div>'
    '<div style="margin:12px 0 4px"><a href="' + html.escape(STUDIO_LINK) + '" target="_blank" '
    'style="display:inline-block;padding:9px 16px;border-radius:10px;background:#c6eda0;color:#26351c;'
    'text-decoration:none;font-weight:600">Open Studio Zero ↗</a>'
    '<span style="font-size:12px;opacity:.7;margin-left:10px">opens the studio on this computer (npm run dev) and connects</span></div>'
    '<div style="font-size:12px;opacity:.7;margin-top:12px">Or paste these into the studio under <b>Connect</b>:</div>'
    + copy_row('Address', SERVICE_URL, 'zero-address') + copy_row('Access key', ACCESS_KEY, 'zero-key') +
    '<div style="font-size:12px;opacity:.7;margin-top:12px">Keep this tab open while you create.</div></div>'))
print('Address:', SERVICE_URL)
print('Access key:', ACCESS_KEY)
LAST_SETUP_AT = time.time()


## Check (optional)
Set **CHECKS** in Settings: *quick* makes one image, *full* tries every feature you ticked and shows the results
together at the end. You can create in the studio while this runs.

In [ ]:
#@title 6 · Check that everything works { display-mode: "form" }
import base64, io, wave
from PIL import Image as PILImage
from IPython.display import HTML, display
results, gallery = [], []

def data_url(raw, kind='image/png'):
    return f'data:{kind};base64,' + base64.b64encode(raw).decode()
def output(job):
    return api.get('/outputs/' + job['output_id'], timeout=300).raise_for_status().content
def clip(text, size=110):
    text = ' '.join(text.split())
    return f'“{text[:size]}…”' if len(text) > size else f'“{text}”'
def sound(raw):
    return 'audio/wav' if raw[:4] == b'RIFF' else 'audio/mpeg'
def png(image):
    buffer = io.BytesIO(); image.save(buffer, 'PNG'); return buffer.getvalue()
def show(label, raw, kind='image/png'):
    gallery.append((label, data_url(raw, kind), kind))
def check(name, work, needed=False):
    start = time.monotonic()
    try:
        detail = work() or ''
    except Exception as exc:
        results.append(False)
        print(f'✗ {name}: {exc}')
        if needed:
            raise
        return
    results.append(True)
    print(f'✓ {name} · {time.monotonic() - start:.1f}s' + (f' · {detail}' if detail else ''))

caps = api.get('/health').raise_for_status().json()['capabilities']
vase = {}
def first_image():
    job, _ = generate({'prompt': 'A small ceramic vase on a wooden table, soft daylight', 'width': 512, 'height': 512,
                       'steps': 8, 'guidance': 6, 'seed': 42})
    vase['raw'] = output(job); vase['url'] = data_url(vase['raw'])
    show('Image', vase['raw'])
    return '512 × 512'

started = time.monotonic()
try:
    if CHECKS == 'off':
        print('Checks are off (CHECKS in Settings).')
    else:
        check('Image', first_image, needed=True)
    if CHECKS == 'full':
        check('Next image (model stays loaded)', lambda: generate(
            {'prompt': 'A paper lantern at dusk', 'width': 512, 'height': 512, 'steps': 8, 'guidance': 6, 'seed': 7}) and '')

        def transparent():
            job, _ = generate({'prompt': 'A small red potion bottle, game item', 'width': 512, 'height': 512, 'steps': 8,
                               'guidance': 6, 'seed': 11, 'transparent': True})
            potion = PILImage.open(io.BytesIO(output(job))).convert('RGBA')
            low, _ = potion.getchannel('A').getextrema()
            assert low < 255, 'the image has no transparent pixels'
            board = PILImage.new('RGBA', potion.size, (60, 60, 60, 255)); board.alpha_composite(potion)
            show('Transparent', png(board))
        check('Transparent image', transparent)

        def inpaint():
            half = PILImage.new('L', (512, 512), 0); half.paste(255, (256, 0, 512, 512))
            job, _ = generate({'operation': 'inpaint', 'prompt': 'a sprig of lavender in the vase', 'width': 512, 'height': 512,
                               'steps': 8, 'guidance': 6, 'seed': 5, 'image': vase['url'], 'mask': data_url(png(half))}, '/edit', 120)
            show('Inpaint (right half)', output(job))
        check('Inpaint', inpaint)

        if 'reference' in caps:
            def reference():
                job, _ = generate({'prompt': 'Change the ceramic vase in image 1 to cobalt blue. Keep the table and lighting.',
                                   'width': 512, 'height': 512, 'steps': 8, 'guidance': 6, 'seed': 42,
                                   'reference_images': [vase['url']]})
                show('Reference edit: blue vase', output(job))
            check('Reference edit', reference)

        if ENABLE_BACKGROUND_REMOVAL:
            def background():
                t1 = time.monotonic()
                while (health := api.get('/health').raise_for_status().json()).get('matting') != 'ready':
                    if health.get('matting') == 'failed' or time.monotonic() - t1 > 600:
                        raise RuntimeError('the model did not load: ' + str(health.get('matting')))
                    time.sleep(1)
                task = api.post('/remove-background', json={'image': vase['url']}).raise_for_status().json()
                while task['status'] in ('queued', 'running'):
                    time.sleep(0.5)
                    task = api.get('/tasks/' + task['id']).raise_for_status().json()
                assert task['status'] == 'succeeded', task['message']
                matte = PILImage.open(io.BytesIO(api.get(f"/tasks/{task['id']}/mask").raise_for_status().content))
                cutout = PILImage.open(io.BytesIO(vase['raw'])).convert('RGBA'); cutout.putalpha(matte)
                board = PILImage.new('RGBA', cutout.size, (40, 40, 40, 255)); board.alpha_composite(cutout)
                show('Background removed', png(board))
                device = health.get('matting_device')
                return f'on {device}' + ('' if device == 'cuda' else ' (slower; check ONNX Runtime CUDA in service.log)')
            check('Background removal', background)

        if 'enhance-prompt' in caps:
            def improve():
                better, _ = run_task('/enhance-prompt', {'prompt': 'a fox in the snow', 'target': 'image'})
                return clip(better['prompt'])
            check('Improve prompt', improve)
        if 'describe' in caps:
            def describe():
                described, _ = run_task('/describe', {'image': vase['url'], 'purpose': 'image'})
                return clip(described['prompt'])
            check('Describe image', describe)

        voice = {}
        if 'speech' in caps:
            def speech():
                job, _ = generate({'text': 'Studio Zero can talk now. This short sentence checks the voice model, '
                                           'and it should sound clear and natural.', 'seed': 3}, '/speech', 120)
                raw = output(job); voice['url'] = data_url(raw, sound(raw))
                show('Voice', raw, sound(raw))
                return 'includes loading the voice model'
            check('Voice', speech)
            if voice:
                def clone():
                    job, _ = generate({'text': 'And this second line reuses the first voice as a sample.',
                                       'voice': voice['url'], 'seed': 4}, '/speech', 120)
                    raw = output(job); show('Cloned voice (should match)', raw, sound(raw))
                check('Voice cloning', clone)
        if 'transcribe' in caps:
            def transcribe():
                sample = voice.get('url')
                if sample is None:  # no voice model: a second of silence still exercises the model
                    silent = io.BytesIO()
                    with wave.open(silent, 'wb') as w:
                        w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(bytes(32000))
                    sample = data_url(silent.getvalue(), 'audio/wav')
                heard, _ = run_task('/transcribe', {'audio': sample})
                return f"on {heard['device']} · " + clip(heard['text'], 160)
            check('Transcription', transcribe)
        if 'music' in caps:
            def music():
                job, _ = generate({'style': 'Genre: indie pop. BPM: 100. Warm female vocal, acoustic guitar and light drums.',
                                   'lyrics': '[verse]\nMorning light on the window\n[chorus]\nHere we go again',
                                   'duration': 20, 'seed': 7}, '/music', 120)
                raw = output(job); show('Music (20 s)', raw, sound(raw))
                return 'includes loading the music model'
            check('Music', music)
        if 'upscale' in caps:
            def upscale():
                job, _ = generate({'image': vase['url'], 'scale': 2}, '/upscale', 120)
                return f"{job['metadata']['source_width']} → {job['metadata']['width']} px wide"
            check('Upscale ×2', upscale)
        if 'detect-pose' in caps:
            from backend.vision import draw_pose
            pose = {}
            def detect():
                job, _ = generate({'prompt': 'Full-body photo of a dancer leaping with arms raised, plain studio background',
                                   'width': 512, 'height': 768, 'steps': 8, 'guidance': 6, 'seed': 21})
                found, _ = run_task('/detect-pose', {'image': data_url(output(job))})
                assert found['people'], 'no person found'
                pose['skeleton'] = draw_pose(found['people'][:1], found['width'], found['height'])
                show('Detected pose', png(pose['skeleton']))
                return f"on {found['device']} · {len(found['people'])} person(s)"
            check('Pose detection', detect)
            if pose and 'reference' in caps:
                def posed():
                    job, _ = generate({'prompt': 'A knight in silver armour in a castle courtyard. The knight takes exactly the '
                                                 'body pose of the coloured stick figure in image 1; do not draw the stick figure.',
                                       'width': 512, 'height': 768, 'steps': 8, 'guidance': 6, 'seed': 5,
                                       'reference_images': [data_url(png(pose['skeleton']))]})
                    show('Knight in that pose', output(job))
                    return 'close to the skeleton, not exact'
                check('Posed image', posed)
        if 'video' in caps:
            def video():
                job, _ = generate({'prompt': 'A paper lantern swaying gently in the wind at dusk, soft light', 'width': 512,
                                   'height': 512, 'frames': 25, 'fps': 24, 'steps': VIDEO_STEPS, 'guidance': VIDEO_GUIDANCE,
                                   'seed': 3}, '/video', 120)
                show('Video', output(job), f'video/{VIDEO_FORMAT}')
                return 'includes swapping in the video model; the image model reloads on the next image'
            check('Video', video)
finally:
    LAST_SETUP_AT = time.time()

if gallery:
    tiles = []
    for label, url, kind in gallery:
        media = (f'<img src="{url}" style="width:100%;border-radius:8px;display:block">' if kind.startswith('image') else
                 f'<audio controls src="{url}" style="width:100%"></audio>' if kind.startswith('audio') else
                 f'<video controls loop muted src="{url}" style="width:100%;border-radius:8px"></video>')
        tiles.append(f'<figure style="margin:0">{media}<figcaption style="font-size:12px;opacity:.75;margin-top:4px">'
                     f'{label}</figcaption></figure>')
    display(HTML('<div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(180px,1fr));gap:14px;'
                 'max-width:980px;font-family:system-ui,sans-serif;margin-top:8px">' + ''.join(tiles) + '</div>'))
if results:
    passed, total = sum(results), len(results)
    seconds = time.monotonic() - started
    took = f'{seconds:.0f}s' if seconds < 90 else f'{seconds / 60:.1f} min'
    if passed == total:
        ok(f'All {total} checks passed in {took}.' if total > 1 else
           'Quick check passed. Set CHECKS to full in Settings to try every feature.')
    else:
        raise RuntimeError(f'{total - passed} of {total} checks failed (✗ above). Details: /content/studio-zero/service.log')


## When you finish

In [ ]:
#@title ⏹ Stop and save (run this on its own when you finish) { display-mode: "form" }
#@markdown Save your work in the studio first (a `.zero` project keeps every image). This stops the engine,
#@markdown waits for any models still being saved to Drive, and flushes Drive so the next session downloads nothing.
import os, subprocess, time
if time.time() - globals().get('LAST_SETUP_AT', 0) < 15:
    print('Skipped: Run all reached this cell. Your engine keeps running; run this cell on its own when you finish.')
else:
    for name in ['tunnel_process', 'server_process']:
        process = globals().get(name)
        if process and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill(); process.wait()
    os.environ.pop('ZERO_API_TOKEN', None)
    print('✓ Engine stopped. The access key no longer works.')
    store = globals().get('model_store')
    if store and store.saving:
        print('• Finishing the Drive save (first session only)…')
        while not store.wait(30):
            print('  still saving to Drive…')
    if store:
        print('✓ Model cache:', store.summary())
    if globals().get('CACHE'):
        from google.colab import drive
        drive.flush_and_unmount()
        print('✓ Drive flushed: the next session reuses everything.')
    print('You can now close the runtime: Runtime → Disconnect and delete runtime.')


## Reference

**Engines.** With `ENGINE = auto`, GPUs with 22 GB or more and runtimes with 45 GB or more of RAM (L4 High-RAM,
A100, H100) use the **fast engine**: Qwen-Image 2.1 in diffusers with Alibaba's Fun-Acc adapter, 4 steps per image
instead of about 28, using the same uncensored UC weights. On an A100 an image or edit takes about 3–5 seconds.
Smaller GPUs use stable-diffusion.cpp, which is compiled once per GPU type and cached on Drive. Every edit mode works
on both. Images over about 2.4 MP decode in tiles so 2K output fits in less memory.

**Drive cache.** Models are saved under *My Drive/StudioZero* in the first session and loaded from there afterwards,
with no Hugging Face downloads. A copy cut short by a runtime reset is detected by size and fetched again. Cached
models stay at the version you first downloaded until you tick `UPDATE_MODELS`. In the first session, run
**Stop and save** before ending the runtime so Drive finishes saving. If Drive is full, sessions still work but
download again.

| Feature | Model | Size | Licence |
|---|---|---|---|
| Images and edits | [Qwen-Image 2.1 UC](https://huggingface.co/abenzerps/Qwen-Image-2.1-GGUF) | 4.6–14 GB + 19 GB companion (fast engine) | Qwen research licence (non-commercial) |
| Remove background | BiRefNet (ONNX) | 1 GB | MIT |
| Improve / Describe | Qwen3-VL-2B Instruct | 4.3 GB | Apache-2.0 |
| Upscale | Real-ESRGAN x4plus | 67 MB | BSD-3 |
| Pose | DWPose + YOLOX | 240 MB | Apache-2.0 |
| Voice | Chatterbox Turbo | 3 GB | MIT (clips carry an inaudible watermark) |
| Transcribe | faster-whisper base | 145 MB | MIT |
| Music | MiniMax Music 3 | 28 GB | Show "MiniMax-Music3" in commercial products; disclose AI music |
| Video | [LTX-2.5](https://huggingface.co/Lightricks/LTX-2.5) distilled Q4_K_M | 44 GB | LTX licence (gated) |

**Notes on features**
- *Improve prompt* can use DeepSeek's API instead of the local model: add a Colab secret named `DEEPSEEK_API_KEY`.
  DeepSeek V4 Flash itself (about 167 GB) is too large to run here. Describe always runs on this runtime.
- *Voice* clones from a clean sample of 10 seconds or more. Only clone voices you have permission to use.
- *Poses* guide images through Qwen-Image's reference pictures, so results follow the pose closely but not exactly.
- *Video* needs the gated Lightricks files: accept the licence on Hugging Face, then paste a read token into
  `HF_TOKEN` in Settings (or add it as the Colab secret `HF_TOKEN`). A token typed into Settings is saved in the
  notebook, so clear it before sharing; the secret is not. Only one big model is in GPU memory at a time, so the first video after images (and the first
  image after a video) takes a minute or two to swap.
- Helper models load on first use. On GPUs with 40 GB or more they stay loaded; on smaller GPUs they unload after use.

**Troubleshooting.** Logs are in `/content/studio-zero`: `service.log` (the API), `outputs/engine.log` (the image
engine) and `tunnel.log`. If loading from Drive stalls, tick `COPY_WEIGHTS_TO_LOCAL`. If attention errors appear,
untick `SAGE_ATTENTION` or `FLASH_ATTENTION` under Advanced settings.

This notebook is generated by `python scripts/build_notebook.py` in the Studio Zero repository.